In [1322]:
import pandas as pd
import psycopg2
import pandas as pd
import geopandas as gpd
import requests
from geopandas.tools import sjoin
import time
import datetime
from collections import Counter 
import re
import random
import numpy as np
import unidecode
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots 
from shapely import wkt
from shapely.geometry import Point

# IMPORTATION DES DONNEES

In [1323]:
#données de patients en idf
df_patients = pd.read_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_patho.csv", sep=";")
df_patients_count = pd.read_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_counts.csv", sep=";")

#Les données de patients avec les pathologies en IDF
gdf_patients_patho_idf = gpd.GeoDataFrame(pd.read_csv('Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_patho.csv',sep=';'), 
                       geometry=gpd.points_from_xy(pd.read_csv('Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_patho.csv',sep=';')['x'], 
                                                   pd.read_csv('Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_patho.csv',sep=';')['y']),
                       crs="EPSG:4326")

#import du shapefile des departements d'IDF
gdf_dept = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/departements/IDF/dpt_idf.shp')
gdf_dept.drop(columns=['patient_co'], axis= 1, inplace=True)


#transformation dans un meme systeme de coordonnées
gdf_patients_patho_idf['CODE_DEPT']= gdf_patients_patho_idf['CODE_DEPT'].astype(str)
gdf_patients_patho_idf = gdf_patients_patho_idf.to_crs(gdf_dept.crs)


#transformation dans un meme systeme de coordonnées
gdf_patients_patho_idf['CODE_DEPT']= gdf_patients_patho_idf['CODE_DEPT'].astype(str)
gdf_dept['CODE_DEPT'] = gdf_dept['CODE_DEPT'].astype(str)
df_patients_count ['CODE_DEPT'] = df_patients_count ['CODE_DEPT'].astype(str)
gdf_patients_patho_idf = gdf_patients_patho_idf.to_crs(gdf_dept.crs)

#mise a jour du nopmbre de patients par departements
patients_counts_by_dept = gdf_dept[['CODE_DEPT','NOM_DEPT']].merge(df_patients_count, on = 'CODE_DEPT')
patients_counts_by_dept.drop(columns=['Unnamed: 0'], axis = 1, inplace= True)

patients_counts_by_dept = patients_counts_by_dept.sort_values(by='CODE_DEPT')
patients_counts_by_dept






,CODE_DEPT,NOM_DEPT,patient_co
3,75,PARIS,10892
5,77,SEINE-ET-MARNE,3018
1,78,YVELINES,8628
6,91,ESSONNE,2992
2,92,HAUTS-DE-SEINE,10174
0,93,SEINE-SAINT-DENIS,3204
4,94,VAL-DE-MARNE,3038
7,95,VAL-D'OISE,2824


In [1324]:
gdf_patients_patho_idf['patho'].unique()

array(['Hemato', 'Ophtalmo', 'Uro', 'Sein', 'Gastro', 'Gynéco', 'Sarcome',
       'Thorax', 'ORL', 'Dermato', 'Autre', 'Endocrino', 'Neuro'],
      dtype=object)

# TRAITEMENT DES DONNEES 

In [1325]:
gdf_idf_hemato_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Hemato'][['CODE_DEPT', 'geometry']]
gdf_idf_sein_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Sein'][['CODE_DEPT', 'geometry']]
gdf_idf_uro_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Uro'][['CODE_DEPT', 'geometry']]
gdf_idf_ophtalmo_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Ophtalmo'][['CODE_DEPT', 'geometry']]
gdf_idf_thorax_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Thorax'][['CODE_DEPT', 'geometry']]
gdf_idf_gastro_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Gastro'][['CODE_DEPT', 'geometry']]
gdf_idf_gyneco_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Gynéco'][['CODE_DEPT', 'geometry']]
gdf_idf_sarcome_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Sarcome'][['CODE_DEPT', 'geometry']]
gdf_idf_orl_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'ORL'][['CODE_DEPT', 'geometry']]
gdf_idf_dermato_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Dermato'][['CODE_DEPT', 'geometry']]
gdf_idf_endocrino_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Endocrino'][['CODE_DEPT', 'geometry']]
gdf_idf_neuro_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Neuro'][['CODE_DEPT', 'geometry']]
gdf_idf_autre_points = gdf_patients_patho_idf[gdf_patients_patho_idf['patho'] == 'Autre'][['CODE_DEPT', 'geometry']]



gdf_idf_thorax_points.columns

Index(['CODE_DEPT', 'geometry'], dtype='object')

In [1326]:
# Fonction pour filtrer et grouper les données par pathologie
def filter_and_group_by_patho(gdf, patho, column_name):
    gdf_patho = gdf[gdf['patho'] == patho]
    gdf_patho_grouped = gdf_patho.groupby('CODE_DEPT').size().reset_index(name=column_name)
    return gdf_patho_grouped

In [1327]:
gdf_idf_hemato = filter_and_group_by_patho(gdf_patients_patho_idf, 'Hemato', 'nb_patients_hemato')
gdf_idf_sein = filter_and_group_by_patho(gdf_patients_patho_idf, 'Sein', 'nb_patients_sein')
gdf_idf_uro = filter_and_group_by_patho(gdf_patients_patho_idf, 'Uro', 'nb_patients_uro')
gdf_idf_ophtalmo = filter_and_group_by_patho(gdf_patients_patho_idf, 'Ophtalmo', 'nb_patients_ophtalmo')
gdf_idf_thorax = filter_and_group_by_patho(gdf_patients_patho_idf, 'Thorax', 'nb_patients_thorax')
gdf_idf_gastro = filter_and_group_by_patho(gdf_patients_patho_idf, 'Gastro', 'nb_patients_gastro')
gdf_idf_gyneco = filter_and_group_by_patho(gdf_patients_patho_idf, 'Gynéco', 'nb_patients_gyneco')
gdf_idf_sarcome = filter_and_group_by_patho(gdf_patients_patho_idf, 'Sarcome', 'nb_patients_sarcome')
gdf_idf_orl = filter_and_group_by_patho(gdf_patients_patho_idf, 'ORL', 'nb_patients_orl')
gdf_idf_dermato = filter_and_group_by_patho(gdf_patients_patho_idf, 'Dermato', 'nb_patients_dermato')
gdf_idf_endocrino = filter_and_group_by_patho(gdf_patients_patho_idf, 'Endocrino', 'nb_patients_endocrino')
gdf_idf_neuro = filter_and_group_by_patho(gdf_patients_patho_idf, 'Neuro', 'nb_patients_neuro')
gdf_idf_autre = filter_and_group_by_patho(gdf_patients_patho_idf, 'Autre', 'nb_patients_autre')


gdf_idf_thorax

,CODE_DEPT,nb_patients_thorax
0,75,487
1,77,123
2,78,184
3,91,62
4,92,352
5,93,102
6,94,131
7,95,76


# STATISTIQUES EN FONCTION DE LA PROXIMITE AUX VOIES FERREES EN IDF

### DANS UN PERIMETRE DE 150M 

#### 1- Repartition de l'ensemble des patients dans cette zone

In [1328]:
df_patients_150m = pd.read_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_150m.csv", sep=";")
df_patients_150m.columns

Index(['field_1', 'pseudo_pro', 'adresse', 'codepost', 'nom_commun', 'requete',
       'x', 'y', 'score', 'trust_scor', 'street', 'city', 'pc_city', 'ic_city',
       'code_dept', 'dept', 'reg', 'address', 'address_ha', 'address__1',
       'same_city', 'hostel', 'hosted', 'date_geolo', 'geometry', 'CODE_IRIS',
       'INSEE_REG', 'CODE_DEPT_', 'patient_se', 'date_naiss', 'centre',
       'ageaudiag', 'cancernum', 'date_diag', 'topo_initi', 'topo_ini_1'],
      dtype='object')

In [1329]:
gdf_buffer_150m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/voies_ferrees_idf/zone_150m.shp')
gdf_buffer_150m = gdf_buffer_150m.to_crs(gdf_patients_patho_idf.crs)
gdf_buffer_150m.columns

Index(['ID', 'ETAT', 'NATURE', 'ELECTRIFIE', 'NB_VOIES', 'LARGEUR', 'POS_SOL',
       'ID_VFN', 'TOPONYME', 'layer', 'path', 'geometry'],
      dtype='object')

##### Pourcentage de la patientèle sur l'ensemble de la région d'IDF

In [1330]:
pourcentage_150m = ((len(df_patients_150m.drop_duplicates(subset = ['pseudo_pro'])))/len(df_patients.drop_duplicates(subset=['pseudo_provisoire']))*100)
print(f"{pourcentage_150m:.2f} % de patients sont situés à moins de 150m des voies ferrées en IDF")

14.03 % de patients sont situés à moins de 150m des voies ferrées en IDF


##### Repartition par département

In [1331]:
patients_counts_exp_by_dept = df_patients_150m[["CODE_DEPT_"]].groupby('CODE_DEPT_').size().reset_index(name = 'exposes')
patients_counts_exp_by_dept


,CODE_DEPT_,exposes
0,75,2097
1,77,213
2,78,872
3,91,287
4,92,1737
5,93,328
6,94,406
7,95,340


In [1332]:
patients_counts_by_dept['CODE_DEPT'] = patients_counts_by_dept['CODE_DEPT'].astype(str)
patients_counts_exp_by_dept['CODE_DEPT_'] = patients_counts_exp_by_dept['CODE_DEPT_'].astype(str)

In [1333]:
stats_150m = patients_counts_by_dept.merge(patients_counts_exp_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT_')
stats_150m.drop(columns=['CODE_DEPT_'], inplace=True)
stats_150m.sort_values(by = 'CODE_DEPT')


,CODE_DEPT,NOM_DEPT,patient_co,exposes
0,75,PARIS,10892,2097
1,77,SEINE-ET-MARNE,3018,213
2,78,YVELINES,8628,872
3,91,ESSONNE,2992,287
4,92,HAUTS-DE-SEINE,10174,1737
5,93,SEINE-SAINT-DENIS,3204,328
6,94,VAL-DE-MARNE,3038,406
7,95,VAL-D'OISE,2824,340


In [1334]:
stats_150m['pourcentage_150'] = (((stats_150m['exposes'] / stats_150m['patient_co'])*100).round(1))
stats_150m = stats_150m.sort_values(by ='CODE_DEPT')
stats_150m

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_150
0,75,PARIS,10892,2097,19.3
1,77,SEINE-ET-MARNE,3018,213,7.1
2,78,YVELINES,8628,872,10.1
3,91,ESSONNE,2992,287,9.6
4,92,HAUTS-DE-SEINE,10174,1737,17.1
5,93,SEINE-SAINT-DENIS,3204,328,10.2
6,94,VAL-DE-MARNE,3038,406,13.4
7,95,VAL-D'OISE,2824,340,12.0


In [1335]:

fig = px.bar(stats_150m, x='NOM_DEPT', y='pourcentage_150', title='Patientèle par departement située à une distance maximale de 150m des voies ferrées',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_150': 'patients(%)'})
fig.show()


###

 #### 2 - Repartition par pathologie des patients dans ce perimetre

In [1336]:
gdf_idf_hemato_dept = gdf_idf_hemato.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_sein_dept = gdf_idf_sein.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_uro_dept = gdf_idf_uro.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_ophtamo_dept = gdf_idf_ophtalmo.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_thorax_dept = gdf_idf_thorax.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_gastro_dept = gdf_idf_gastro.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_gyneco_dept = gdf_idf_gyneco.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_sarcome_dept = gdf_idf_sarcome.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_orl_dept = gdf_idf_orl.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_dermato_dept = gdf_idf_dermato.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_endocrino_dept = gdf_idf_endocrino.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_neuro_dept = gdf_idf_neuro.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')
gdf_idf_autre_dept = gdf_idf_autre.merge(gdf_dept[['NOM_DEPT','CODE_DEPT']], on='CODE_DEPT')



gdf_idf_thorax_dept

,CODE_DEPT,nb_patients_thorax,NOM_DEPT
0,75,487,PARIS
1,77,123,SEINE-ET-MARNE
2,78,184,YVELINES
3,91,62,ESSONNE
4,92,352,HAUTS-DE-SEINE
5,93,102,SEINE-SAINT-DENIS
6,94,131,VAL-DE-MARNE
7,95,76,VAL-D'OISE


In [1337]:

gdf_idf_hemato_150m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_hemato_150m_dept = gdf_idf_hemato_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_hemato_150m = gdf_idf_hemato_150m_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_150m['pourcentage']= (gdf_idf_hemato_150m['nbre_patients_<=150m']/gdf_idf_hemato_150m['nb_patients_hemato']*100).round(1)


gdf_idf_sein_150m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_sein_150m_dept = gdf_idf_sein_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_sein_150m = gdf_idf_sein_150m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_150m['pourcentage']= (gdf_idf_sein_150m['nbre_patients_<=150m']/gdf_idf_sein_150m['nb_patients_sein']*100).round(1)


gdf_idf_uro_150m = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_uro_150m_dept = gdf_idf_uro_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_uro_150m = gdf_idf_uro_150m_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_150m['pourcentage']= (gdf_idf_uro_150m['nbre_patients_<=150m']/gdf_idf_uro_150m['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_150m = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_ophtalmo_150m_dept = gdf_idf_ophtalmo_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_ophtalmo_150m = gdf_idf_ophtalmo_150m_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_150m['pourcentage'] = (gdf_idf_ophtalmo_150m['nbre_patients_<=150m']/gdf_idf_ophtalmo_150m['nb_patients_ophtalmo']*100).round(1)


gdf_idf_thorax_150m = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_thorax_150m_dept = gdf_idf_thorax_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_thorax_150m = gdf_idf_thorax_150m_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_150m['pourcentage'] = (gdf_idf_thorax_150m['nbre_patients_<=150m']/gdf_idf_thorax_150m['nb_patients_thorax']*100).round(1)


gdf_idf_gastro_150m = gpd.sjoin(gdf_idf_gastro_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_gastro_150m_dept = gdf_idf_gastro_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_gastro_150m = gdf_idf_gastro_150m_dept.merge(gdf_idf_gastro_dept, on='CODE_DEPT')
gdf_idf_gastro_150m['pourcentage'] = (gdf_idf_gastro_150m['nbre_patients_<=150m']/gdf_idf_gastro_150m['nb_patients_gastro']*100).round(1)

gdf_idf_gyneco_150m = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_gyneco_150m_dept = gdf_idf_gyneco_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_gyneco_150m = gdf_idf_gyneco_150m_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_150m['pourcentage'] = (gdf_idf_gyneco_150m['nbre_patients_<=150m']/gdf_idf_gyneco_150m['nb_patients_gyneco']*100).round(1)

gdf_idf_sarcome_150m = gpd.sjoin(gdf_idf_sarcome_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_sarcome_150m_dept = gdf_idf_sarcome_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_sarcome_150m = gdf_idf_sarcome_150m_dept.merge(gdf_idf_sarcome_dept, on='CODE_DEPT')
gdf_idf_sarcome_150m['pourcentage'] = (gdf_idf_sarcome_150m['nbre_patients_<=150m']/gdf_idf_sarcome_150m['nb_patients_sarcome']*100).round(1)

gdf_idf_orl_150m = gpd.sjoin(gdf_idf_orl_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_orl_150m_dept = gdf_idf_orl_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_orl_150m = gdf_idf_orl_150m_dept.merge(gdf_idf_orl_dept, on='CODE_DEPT')
gdf_idf_orl_150m['pourcentage'] = (gdf_idf_orl_150m['nbre_patients_<=150m']/gdf_idf_orl_150m['nb_patients_orl']*100).round(1)

gdf_idf_dermato_150m = gpd.sjoin(gdf_idf_dermato_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_dermato_150m_dept = gdf_idf_dermato_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_dermato_150m = gdf_idf_dermato_150m_dept.merge(gdf_idf_dermato_dept, on='CODE_DEPT')
gdf_idf_dermato_150m['pourcentage'] = (gdf_idf_dermato_150m['nbre_patients_<=150m']/gdf_idf_dermato_150m['nb_patients_dermato']*100).round(1)

gdf_idf_endocrino_150m = gpd.sjoin(gdf_idf_endocrino_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_endocrino_150m_dept = gdf_idf_endocrino_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_endocrino_150m = gdf_idf_endocrino_150m_dept.merge(gdf_idf_endocrino_dept, on='CODE_DEPT')
gdf_idf_endocrino_150m['pourcentage'] = (gdf_idf_endocrino_150m['nbre_patients_<=150m']/gdf_idf_endocrino_150m['nb_patients_endocrino']*100).round(1)

gdf_idf_neuro_150m = gpd.sjoin(gdf_idf_neuro_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_neuro_150m_dept = gdf_idf_neuro_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_neuro_150m = gdf_idf_neuro_150m_dept.merge(gdf_idf_neuro_dept, on='CODE_DEPT')
gdf_idf_neuro_150m['pourcentage'] = (gdf_idf_neuro_150m['nbre_patients_<=150m']/gdf_idf_neuro_150m['nb_patients_neuro']*100).round(1)

gdf_idf_autre_150m = gpd.sjoin(gdf_idf_autre_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_autre_150m_dept = gdf_idf_autre_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_autre_150m = gdf_idf_autre_150m_dept.merge(gdf_idf_autre_dept, on='CODE_DEPT')
gdf_idf_autre_150m['pourcentage'] = (gdf_idf_autre_150m['nbre_patients_<=150m']/gdf_idf_autre_150m['nb_patients_autre']*100).round(1)



In [1338]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_150m['NOM_DEPT'], y=gdf_idf_hemato_150m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_150m['NOM_DEPT'], y=gdf_idf_sein_150m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_150m['NOM_DEPT'], y=gdf_idf_uro_150m['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_150m['NOM_DEPT'], y=gdf_idf_ophtalmo_150m['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_thorax_150m['NOM_DEPT'], y=gdf_idf_thorax_150m['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_150m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 150m des voies ferrées en fonction de leur groupe pathologique')

fig.show()

In [1339]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_gastro_150m['NOM_DEPT'], y=gdf_idf_gastro_150m['pourcentage'], name='cancer gastro',
                     text=gdf_idf_gastro_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_150m['NOM_DEPT'], y=gdf_idf_gyneco_150m['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sarcome_150m['NOM_DEPT'], y=gdf_idf_sarcome_150m['pourcentage'], name='cancer sarcome',
                     text=gdf_idf_sarcome_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_orl_150m['NOM_DEPT'], y=gdf_idf_orl_150m['pourcentage'], name='cancer orl',
                     text=gdf_idf_orl_150m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 150m des voies ferrées en fonction de leur groupe pathologique')

fig.show()

In [1340]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_dermato_150m['NOM_DEPT'], y=gdf_idf_dermato_150m['pourcentage'], name='cancer dermato',
                     text=gdf_idf_dermato_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_endocrino_150m['NOM_DEPT'], y=gdf_idf_endocrino_150m['pourcentage'], name='cancer endocrino',
                     text=gdf_idf_endocrino_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_neuro_150m['NOM_DEPT'], y=gdf_idf_neuro_150m['pourcentage'], name='cancer neuro',
                     text=gdf_idf_neuro_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_autre_150m['NOM_DEPT'], y=gdf_idf_autre_150m['pourcentage'], name='cancer autre',
                     text=gdf_idf_autre_150m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 150m des voies ferrées en fonction de leur groupe pathologique')

fig.show()

### DANS UN PERIMETRE DE 500M 

#### 1 - Repartition de l'ensemble des patients dans ce périmètre

In [1341]:
df_500m = pd.read_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_500m.csv", sep=";")
patients_counts_exp500_by_dept = df_500m.groupby('CODE_DEPT_').size().reset_index(name='exposes')
patients_counts_exp500_by_dept


,CODE_DEPT_,exposes
0,75,6658
1,77,917
2,78,2969
3,91,1011
4,92,5019
5,93,1274
6,94,1348
7,95,1185


##### Pourcentage sur l'ensemble de la région d'IDF

In [1342]:
len(df_500m.drop_duplicates(subset='pseudo_pro'))

20381

In [1343]:
pourcentage = len(df_500m.drop_duplicates(subset='pseudo_pro'))/len(df_patients.drop_duplicates(subset='pseudo_provisoire'))*100
pourcentage.__round__(2)
print(f"{pourcentage:.2f}% de patients sont situés à moins de 500m des voies ferrées en IDF")

45.52% de patients sont situés à moins de 500m des voies ferrées en IDF


##### Repartition par département

In [1344]:
patients_counts_exp500_by_dept['CODE_DEPT_'] = patients_counts_exp_by_dept['CODE_DEPT_'].astype(str)


In [1345]:
stats_500m = patients_counts_by_dept.merge(patients_counts_exp500_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT_')
stats_500m.drop(columns=['CODE_DEPT_'], inplace=True)
stats_500m

,CODE_DEPT,NOM_DEPT,patient_co,exposes
0,75,PARIS,10892,6658
1,77,SEINE-ET-MARNE,3018,917
2,78,YVELINES,8628,2969
3,91,ESSONNE,2992,1011
4,92,HAUTS-DE-SEINE,10174,5019
5,93,SEINE-SAINT-DENIS,3204,1274
6,94,VAL-DE-MARNE,3038,1348
7,95,VAL-D'OISE,2824,1185


In [1346]:
stats_500m['pourcentage_500'] = (stats_500m['exposes'] / stats_500m['patient_co'])*100
stats_500m = stats_500m.sort_values(by ='CODE_DEPT')
stats_500m

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_500
0,75,PARIS,10892,6658,61.127433
1,77,SEINE-ET-MARNE,3018,917,30.384361
2,78,YVELINES,8628,2969,34.411219
3,91,ESSONNE,2992,1011,33.790107
4,92,HAUTS-DE-SEINE,10174,5019,49.331630
5,93,SEINE-SAINT-DENIS,3204,1274,39.762797
6,94,VAL-DE-MARNE,3038,1348,44.371297
7,95,VAL-D'OISE,2824,1185,41.961756


In [1347]:
fig = px.bar(stats_500m, x='NOM_DEPT', y='pourcentage_500', title='Patientèle située à une distance maximale de 500m des voies ferrées par département',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_500': 'patients (%)'})
fig.show()

#### 2 - Repartition par pathologie des patients dans ce périmètre

In [1348]:
gdf_buffer_500m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/voies_ferrees_idf/zone_500m.shp')
gdf_buffer_500m = gdf_buffer_500m.to_crs(gdf_patients_patho_idf.crs)
gdf_buffer_500m.columns

Index(['ID', 'ETAT', 'NATURE', 'ELECTRIFIE', 'NB_VOIES', 'LARGEUR', 'POS_SOL',
       'ID_VFN', 'TOPONYME', 'layer', 'path', 'geometry'],
      dtype='object')

In [1349]:
gdf_idf_hemato_500m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_hemato_500m_dept = gdf_idf_hemato_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_hemato_500m = gdf_idf_hemato_500m_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_500m['pourcentage']= (gdf_idf_hemato_500m['nbre_patients_<=500m']/gdf_idf_hemato_500m['nb_patients_hemato']*100).round(1)



gdf_idf_sein_500m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_sein_500m_dept = gdf_idf_sein_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_sein_500m = gdf_idf_sein_500m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_500m['pourcentage']= (gdf_idf_sein_500m['nbre_patients_<=500m']/gdf_idf_sein_500m['nb_patients_sein']*100).round(1)


gdf_idf_uro_500m = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_uro_500m_dept = gdf_idf_uro_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_uro_500m = gdf_idf_uro_500m_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_500m['pourcentage']= (gdf_idf_uro_500m['nbre_patients_<=500m']/gdf_idf_uro_500m['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_500m = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_ophtalmo_500m_dept = gdf_idf_ophtalmo_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_ophtalmo_500m = gdf_idf_ophtalmo_500m_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_500m['pourcentage'] = (gdf_idf_ophtalmo_500m['nbre_patients_<=500m']/gdf_idf_ophtalmo_500m['nb_patients_ophtalmo']*100).round(1)

gdf_idf_thorax_500m = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_thorax_500m_dept = gdf_idf_thorax_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_thorax_500m = gdf_idf_thorax_500m_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_500m['pourcentage'] = (gdf_idf_thorax_500m['nbre_patients_<=500m']/gdf_idf_thorax_500m['nb_patients_thorax']*100).round(1)

gdf_idf_gastro_500m = gpd.sjoin(gdf_idf_gastro_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_gastro_500m_dept = gdf_idf_gastro_500m .groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_gastro_500m  = gdf_idf_gastro_500m_dept.merge(gdf_idf_gastro_dept, on='CODE_DEPT')
gdf_idf_gastro_500m ['pourcentage'] = (gdf_idf_gastro_500m['nbre_patients_<=500m']/gdf_idf_gastro_500m['nb_patients_gastro']*100).round(1)

gdf_idf_gyneco_500m  = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_500m , how='inner', predicate='intersects')
gdf_idf_gyneco_500m_dept = gdf_idf_gyneco_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_gyneco_500m = gdf_idf_gyneco_500m_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_500m['pourcentage'] = (gdf_idf_gyneco_500m['nbre_patients_<=500m']/gdf_idf_gyneco_500m['nb_patients_gyneco']*100).round(1)

gdf_idf_sarcome_500m = gpd.sjoin(gdf_idf_sarcome_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_sarcome_500m_dept = gdf_idf_sarcome_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_sarcome_500m  = gdf_idf_sarcome_500m_dept.merge(gdf_idf_sarcome_dept, on='CODE_DEPT')
gdf_idf_sarcome_500m['pourcentage'] = (gdf_idf_sarcome_500m['nbre_patients_<=500m']/gdf_idf_sarcome_500m['nb_patients_sarcome']*100).round(1)

gdf_idf_orl_500m = gpd.sjoin(gdf_idf_orl_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_orl_500m_dept = gdf_idf_orl_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_orl_500m = gdf_idf_orl_500m_dept.merge(gdf_idf_orl_dept, on='CODE_DEPT')
gdf_idf_orl_500m['pourcentage'] = (gdf_idf_orl_500m['nbre_patients_<=500m']/gdf_idf_orl_500m['nb_patients_orl']*100).round(1)

gdf_idf_dermato_500m = gpd.sjoin(gdf_idf_dermato_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_dermato_500m_dept = gdf_idf_dermato_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_dermato_500m= gdf_idf_dermato_500m_dept.merge(gdf_idf_dermato_dept, on='CODE_DEPT')
gdf_idf_dermato_500m['pourcentage'] = (gdf_idf_dermato_500m['nbre_patients_<=500m']/gdf_idf_dermato_500m['nb_patients_dermato']*100).round(1)

gdf_idf_endocrino_500m = gpd.sjoin(gdf_idf_endocrino_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_endocrino_500m_dept = gdf_idf_endocrino_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_endocrino_500m = gdf_idf_endocrino_500m_dept.merge(gdf_idf_endocrino_dept, on='CODE_DEPT')
gdf_idf_endocrino_500m['pourcentage'] = (gdf_idf_endocrino_500m['nbre_patients_<=500m']/gdf_idf_endocrino_500m['nb_patients_endocrino']*100).round(1)

gdf_idf_neuro_500m = gpd.sjoin(gdf_idf_neuro_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_neuro_500m_dept = gdf_idf_neuro_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_neuro_500m  = gdf_idf_neuro_500m_dept.merge(gdf_idf_neuro_dept, on='CODE_DEPT')
gdf_idf_neuro_500m['pourcentage'] = (gdf_idf_neuro_500m['nbre_patients_<=500m']/gdf_idf_neuro_500m['nb_patients_neuro']*100).round(1)

gdf_idf_autre_500m = gpd.sjoin(gdf_idf_autre_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_autre_500m_dept = gdf_idf_autre_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_autre_500m = gdf_idf_autre_500m_dept.merge(gdf_idf_autre_dept, on='CODE_DEPT')
gdf_idf_autre_500m['pourcentage'] = (gdf_idf_autre_500m['nbre_patients_<=500m']/gdf_idf_autre_500m['nb_patients_autre']*100).round(1)


print(gdf_idf_hemato_500m)
print(gdf_idf_sein_500m)
print(gdf_idf_uro_500m)
print(gdf_idf_ophtalmo_500m)


  CODE_DEPT  nbre_patients_<=500m  nb_patients_hemato           NOM_DEPT  \
0        75                   226                 357              PARIS   
1        77                    30                  90     SEINE-ET-MARNE   
2        78                   112                 307           YVELINES   
3        91                    37                 101            ESSONNE   
4        92                   371                 717     HAUTS-DE-SEINE   
5        93                    32                  91  SEINE-SAINT-DENIS   
6        94                    34                  93       VAL-DE-MARNE   
7        95                    27                  95         VAL-D'OISE   

   pourcentage  
0         63.3  
1         33.3  
2         36.5  
3         36.6  
4         51.7  
5         35.2  
6         36.6  
7         28.4  
  CODE_DEPT  nbre_patients_<=500m  nb_patients_sein           NOM_DEPT  \
0        75                  4134              6754              PARIS   
1        77   

In [1350]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_500m['NOM_DEPT'], y=gdf_idf_hemato_500m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_500m['NOM_DEPT'], y=gdf_idf_sein_500m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_500m['NOM_DEPT'], y=gdf_idf_uro_500m['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_500m['NOM_DEPT'], y=gdf_idf_ophtalmo_500m['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_thorax_500m['NOM_DEPT'], y=gdf_idf_thorax_500m['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située à une distance maximale de 500m des voies ferrées par département et en fonction de leur pathologie')

fig.show()

In [1351]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_gastro_500m['NOM_DEPT'], y=gdf_idf_gastro_500m['pourcentage'], name='cancer gastro',
                     text=gdf_idf_gastro_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_500m['NOM_DEPT'], y=gdf_idf_gyneco_500m['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sarcome_500m['NOM_DEPT'], y=gdf_idf_sarcome_500m['pourcentage'], name='cancer sarcome',
                     text=gdf_idf_sarcome_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_orl_500m['NOM_DEPT'], y=gdf_idf_orl_500m['pourcentage'], name='cancer orl',
                     text=gdf_idf_orl_500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 500m des voies ferrées en fonction de leur groupe pathologique')

fig.show()

In [1352]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_dermato_500m['NOM_DEPT'], y=gdf_idf_dermato_500m['pourcentage'], name='cancer dermato',
                     text=gdf_idf_dermato_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_endocrino_500m['NOM_DEPT'], y=gdf_idf_endocrino_500m['pourcentage'], name='cancer endocrino',
                     text=gdf_idf_endocrino_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_neuro_500m['NOM_DEPT'], y=gdf_idf_neuro_500m['pourcentage'], name='cancer neuro',
                     text=gdf_idf_neuro_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_autre_500m['NOM_DEPT'], y=gdf_idf_autre_500m['pourcentage'], name='cancer autre',
                     text=gdf_idf_autre_500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 150m des voies ferrées en fonction de leur groupe pathologique')

fig.show()

### DANS UN PERIMETRE DE 1 KM

#### 1 - Repartition pour l'ensemble des patients dans ce périmètre

In [1353]:
df_1000m = pd.read_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_1km.csv", sep=";")
patient_counts_exp1000m_by_dept = df_1000m.groupby('CODE_DEPT_').size().reset_index(name='exposes')

df_1000m.columns

Index(['field_1', 'pseudo_pro', 'adresse', 'codepost', 'nom_commun', 'requete',
       'x', 'y', 'score', 'trust_scor', 'street', 'city', 'pc_city', 'ic_city',
       'code_dept', 'dept', 'reg', 'address', 'address_ha', 'address__1',
       'same_city', 'hostel', 'hosted', 'date_geolo', 'geometry', 'CODE_IRIS',
       'INSEE_REG', 'CODE_DEPT_', 'patient_se', 'date_naiss', 'centre',
       'ageaudiag', 'cancernum', 'date_diag', 'topo_initi', 'topo_ini_1'],
      dtype='object')

##### Pourcentage dans l'ensemble de la région d'IDF

In [1354]:
len(df_1000m.drop_duplicates(subset='pseudo_pro'))

32906

In [1355]:
pourcentage = len(df_1000m.drop_duplicates(subset='pseudo_pro'))/len(df_patients.drop_duplicates(subset='pseudo_provisoire'))*100
pourcentage.__round__(2)

73.5

##### Repartition par département

In [1356]:
patient_counts_exp1000m_by_dept['CODE_DEPT_'] = patient_counts_exp1000m_by_dept['CODE_DEPT_'].astype(str)

In [1357]:
stats_1000m = patients_counts_by_dept.merge(patient_counts_exp1000m_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT_')
stats_1000m.drop(columns=['CODE_DEPT_'], inplace=True)
stats_1000m = stats_1000m.sort_values(by='CODE_DEPT')


In [1358]:
stats_1000m['pourcentage_1000'] = (stats_1000m['exposes'] / stats_1000m['patient_co'])*100
stats_1000m

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_1000
0,75,PARIS,10892,10127,92.976497
1,77,SEINE-ET-MARNE,3018,1597,52.915838
2,78,YVELINES,8628,5099,59.098285
3,91,ESSONNE,2992,1776,59.358289
4,92,HAUTS-DE-SEINE,10174,7945,78.091213
5,93,SEINE-SAINT-DENIS,3204,2187,68.258427
6,94,VAL-DE-MARNE,3038,2241,73.765635
7,95,VAL-D'OISE,2824,1934,68.484419


In [1359]:
fig = px.bar(stats_1000m, x='NOM_DEPT', y='pourcentage_1000', title='Patientèle située à une distance maximale de 1km des voies ferrées par departement',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_1000': 'patients (%)'})
fig.show()

#### 2 - Repartition par pathologie des patients dans ce périmetre

In [1360]:
gdf_buffer_1km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/voies_ferrees_idf/zone_1km.shp')
gdf_buffer_1km = gdf_buffer_1km.to_crs(gdf_patients_patho_idf.crs)


In [1361]:
gdf_idf_hemato_1km = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_hemato_1km_dept = gdf_idf_hemato_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_hemato_1km = gdf_idf_hemato_1km_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_1km['pourcentage']= (gdf_idf_hemato_1km['nbre_patients_<=1km']/gdf_idf_hemato_1km['nb_patients_hemato']*100).round(1)



gdf_idf_sein_1km = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_sein_1km_dept = gdf_idf_sein_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_sein_1km = gdf_idf_sein_1km_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_1km['pourcentage']= (gdf_idf_sein_1km['nbre_patients_<=1km']/gdf_idf_sein_1km['nb_patients_sein']*100).round(1)


gdf_idf_uro_1km = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_uro_1km_dept = gdf_idf_uro_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_uro_1km = gdf_idf_uro_1km_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_1km['pourcentage']= (gdf_idf_uro_1km['nbre_patients_<=1km']/gdf_idf_uro_1km['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_1km = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_ophtalmo_1km_dept = gdf_idf_ophtalmo_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_ophtalmo_1km = gdf_idf_ophtalmo_1km_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_1km['pourcentage'] = (gdf_idf_ophtalmo_1km['nbre_patients_<=1km']/gdf_idf_ophtalmo_1km['nb_patients_ophtalmo']*100).round(1)

gdf_idf_thorax_1km = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_thorax_1km_dept = gdf_idf_thorax_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_thorax_1km = gdf_idf_thorax_1km_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_1km['pourcentage'] = (gdf_idf_thorax_1km['nbre_patients_<=1km']/gdf_idf_thorax_500m['nb_patients_thorax']*100).round(1)


gdf_idf_gastro_1km = gpd.sjoin(gdf_idf_gastro_points, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_idf_gastro_1km_dept = gdf_idf_gastro_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_gastro_1km  = gdf_idf_gastro_1km_dept.merge(gdf_idf_gastro_dept, on='CODE_DEPT')
gdf_idf_gastro_1km ['pourcentage'] = (gdf_idf_gastro_1km['nbre_patients_<=1km']/gdf_idf_gastro_1km['nb_patients_gastro']*100).round(1)

gdf_idf_gyneco_1km  = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_1km , how='inner', predicate='intersects')
gdf_idf_gyneco_1km_dept = gdf_idf_gyneco_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_gyneco_1km = gdf_idf_gyneco_1km_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_1km['pourcentage'] = (gdf_idf_gyneco_1km['nbre_patients_<=1km']/gdf_idf_gyneco_1km['nb_patients_gyneco']*100).round(1)

gdf_idf_sarcome_1km = gpd.sjoin(gdf_idf_sarcome_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_sarcome_1km_dept = gdf_idf_sarcome_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_sarcome_1km = gdf_idf_sarcome_1km_dept.merge(gdf_idf_sarcome_dept, on='CODE_DEPT')
gdf_idf_sarcome_1km['pourcentage'] = (gdf_idf_sarcome_1km['nbre_patients_<=1km']/gdf_idf_sarcome_1km['nb_patients_sarcome']*100).round(1)

gdf_idf_orl_1km = gpd.sjoin(gdf_idf_orl_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_orl_1km_dept = gdf_idf_orl_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_orl_1km = gdf_idf_orl_1km_dept.merge(gdf_idf_orl_dept, on='CODE_DEPT')
gdf_idf_orl_1km['pourcentage'] = (gdf_idf_orl_1km['nbre_patients_<=1km']/gdf_idf_orl_1km['nb_patients_orl']*100).round(1)

gdf_idf_dermato_1km = gpd.sjoin(gdf_idf_dermato_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_dermato_1km_dept = gdf_idf_dermato_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_dermato_1km= gdf_idf_dermato_1km_dept.merge(gdf_idf_dermato_dept, on='CODE_DEPT')
gdf_idf_dermato_1km['pourcentage'] = (gdf_idf_dermato_1km['nbre_patients_<=1km']/gdf_idf_dermato_1km['nb_patients_dermato']*100).round(1)

gdf_idf_endocrino_1km = gpd.sjoin(gdf_idf_endocrino_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_endocrino_1km_dept = gdf_idf_endocrino_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_endocrino_1km = gdf_idf_endocrino_1km_dept.merge(gdf_idf_endocrino_dept, on='CODE_DEPT')
gdf_idf_endocrino_1km['pourcentage'] = (gdf_idf_endocrino_1km['nbre_patients_<=1km']/gdf_idf_endocrino_1km['nb_patients_endocrino']*100).round(1)

gdf_idf_neuro_1km = gpd.sjoin(gdf_idf_neuro_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_neuro_1km_dept = gdf_idf_neuro_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_neuro_1km  = gdf_idf_neuro_1km_dept.merge(gdf_idf_neuro_dept, on='CODE_DEPT')
gdf_idf_neuro_1km['pourcentage'] = (gdf_idf_neuro_1km['nbre_patients_<=1km']/gdf_idf_neuro_1km['nb_patients_neuro']*100).round(1)

gdf_idf_autre_1km = gpd.sjoin(gdf_idf_autre_points, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_idf_autre_1km_dept = gdf_idf_autre_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_autre_1km = gdf_idf_autre_1km_dept.merge(gdf_idf_autre_dept, on='CODE_DEPT')
gdf_idf_autre_1km['pourcentage'] = (gdf_idf_autre_1km['nbre_patients_<=1km']/gdf_idf_autre_1km['nb_patients_autre']*100).round(1)

print(gdf_idf_dermato_1km)
print(gdf_idf_endocrino_1km)
print(gdf_idf_neuro_1km)
print(gdf_idf_autre_1km)

  CODE_DEPT  nbre_patients_<=1km  nb_patients_dermato           NOM_DEPT  \
0        75                  218                  231              PARIS   
1        77                   24                   45     SEINE-ET-MARNE   
2        78                   98                  154           YVELINES   
3        91                   39                   66            ESSONNE   
4        92                  132                  169     HAUTS-DE-SEINE   
5        93                   27                   45  SEINE-SAINT-DENIS   
6        94                   42                   63       VAL-DE-MARNE   
7        95                   34                   47         VAL-D'OISE   

   pourcentage  
0         94.4  
1         53.3  
2         63.6  
3         59.1  
4         78.1  
5         60.0  
6         66.7  
7         72.3  
  CODE_DEPT  nbre_patients_<=1km  nb_patients_endocrino           NOM_DEPT  \
0        75                   74                     78              PARIS   
1     

In [1362]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_1km['NOM_DEPT'], y=gdf_idf_hemato_1km['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_1km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_1km['NOM_DEPT'], y=gdf_idf_sein_1km['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_1km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_1km['NOM_DEPT'], y=gdf_idf_uro_1km['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_1km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_1km['NOM_DEPT'], y=gdf_idf_ophtalmo_1km['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_1km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_thorax_1km['NOM_DEPT'], y=gdf_idf_thorax_1km['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_1km['pourcentage'].round(1),
                     textposition='outside'))



fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située à une distance maximale d\'1km des voies ferrées par département et en fonction de leur pathologie')

fig.show()

In [1363]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_gastro_1km['NOM_DEPT'], y=gdf_idf_gastro_1km['pourcentage'], name='cancer gastro',
                     text=gdf_idf_gastro_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_1km['NOM_DEPT'], y=gdf_idf_gyneco_1km['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sarcome_1km['NOM_DEPT'], y=gdf_idf_sarcome_1km['pourcentage'], name='cancer sarcome',
                     text=gdf_idf_sarcome_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_orl_1km['NOM_DEPT'], y=gdf_idf_orl_1km['pourcentage'], name='cancer orl',
                     text=gdf_idf_orl_1km['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 1km des voies ferrées en fonction de leur groupe pathologique')

fig.show()

In [1364]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_dermato_1km['NOM_DEPT'], y=gdf_idf_dermato_1km['pourcentage'], name='cancer dermato',
                     text=gdf_idf_dermato_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_endocrino_1km['NOM_DEPT'], y=gdf_idf_endocrino_1km['pourcentage'], name='cancer endocrino',
                     text=gdf_idf_endocrino_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_neuro_1km['NOM_DEPT'], y=gdf_idf_neuro_1km['pourcentage'], name='cancer neuro',
                     text=gdf_idf_neuro_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_autre_1km['NOM_DEPT'], y=gdf_idf_autre_1km['pourcentage'], name='cancer autre',
                     text=gdf_idf_autre_1km['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 1km des voies ferrées en fonction de leur groupe pathologique')

fig.show()

### PERIMETRE DE 1.5 KM

#### 1 - Repartition de l'ensemble de patients dans cette zone

In [1365]:
df_1500m = pd.read_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_1500m.csv", sep=";")
patient_counts_exp1500m_by_dept = df_1500m.groupby('CODE_DEPT_').size().reset_index(name='exposes')

df_1500m.columns

Index(['field_1', 'pseudo_pro', 'adresse', 'codepost', 'nom_commun', 'requete',
       'x', 'y', 'score', 'trust_scor', 'street', 'city', 'pc_city', 'ic_city',
       'code_dept', 'dept', 'reg', 'address', 'address_ha', 'address__1',
       'same_city', 'hostel', 'hosted', 'date_geolo', 'geometry', 'CODE_IRIS',
       'INSEE_REG', 'CODE_DEPT_', 'patient_se', 'date_naiss', 'centre',
       'ageaudiag', 'cancernum', 'date_diag', 'topo_initi', 'topo_ini_1'],
      dtype='object')

##### Pourcentage sur l'ensemble de la région d'IDF

In [1366]:
len(df_1500m.drop_duplicates(subset='pseudo_pro'))

38693

In [1367]:
pourcentage = len(df_1500m.drop_duplicates(subset='pseudo_pro'))/len(df_patients.drop_duplicates(subset='pseudo_provisoire'))*100
pourcentage.__round__(2)

86.42

In [1368]:
patient_counts_exp1500m_by_dept['CODE_DEPT_'] = patient_counts_exp1500m_by_dept['CODE_DEPT_'].astype(str)

In [1369]:
stats_1500m = patients_counts_by_dept.merge(patient_counts_exp1500m_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT_')
stats_1500m.drop(columns=['CODE_DEPT_'], inplace=True)
stats_1500m = stats_1500m.sort_values(by = 'CODE_DEPT')

In [1370]:
stats_1500m['pourcentage_1500'] = (stats_1500m['exposes'] / stats_1500m['patient_co'])*100
stats_1500m

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_1500
0,75,PARIS,10892,10885,99.935733
1,77,SEINE-ET-MARNE,3018,2058,68.190855
2,78,YVELINES,8628,6326,73.319425
3,91,ESSONNE,2992,2231,74.565508
4,92,HAUTS-DE-SEINE,10174,9412,92.510320
5,93,SEINE-SAINT-DENIS,3204,2728,85.143571
6,94,VAL-DE-MARNE,3038,2675,88.051350
7,95,VAL-D'OISE,2824,2378,84.206799


In [1371]:
fig = px.bar(stats_1500m, x='NOM_DEPT', y='pourcentage_1500', title='patientèle située a une distance maximale d\'1.5km des voies ferrées par departement',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_1500': 'patients (%)'})
fig.show()

#### 2 - Repartition par pathologie des patients dans cette zone

In [1372]:
gdf_buffer_1_5km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/voies_ferrees_idf/zone_1.5.shp')
gdf_buffer_1_5km = gdf_buffer_1_5km.to_crs(gdf_patients_patho_idf.crs)

In [1373]:
gdf_idf_hemato_1500m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_1_5km, how='inner', predicate='intersects')
gdf_idf_hemato_1500m_dept = gdf_idf_hemato_1500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1500m')
gdf_idf_hemato_1500m = gdf_idf_hemato_1500m_dept .merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_1500m['pourcentage']= (gdf_idf_hemato_1500m['nbre_patients_<=1500m']/gdf_idf_hemato_1500m['nb_patients_hemato']*100).round(1)



gdf_idf_sein_1500m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_1_5km, how='inner', predicate='intersects')
gdf_idf_sein_1500m_dept = gdf_idf_sein_1500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1500m')
gdf_idf_sein_1500m = gdf_idf_sein_1500m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_1500m['pourcentage']= (gdf_idf_sein_1500m['nbre_patients_<=1500m']/gdf_idf_sein_1500m['nb_patients_sein']*100).round(1)


gdf_idf_uro_1500m = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_1_5km, how='inner', predicate='intersects')
gdf_idf_uro_1500m_dept = gdf_idf_uro_1500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1500m')
gdf_idf_uro_1500m = gdf_idf_uro_1500m_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_1500m['pourcentage']= (gdf_idf_uro_1500m['nbre_patients_<=1500m']/gdf_idf_uro_1500m['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_1500m = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_1_5km, how='inner', predicate='intersects')
gdf_idf_ophtalmo_1500m_dept = gdf_idf_ophtalmo_1500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1500m')
gdf_idf_ophtalmo_1500m = gdf_idf_ophtalmo_1500m_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_1500m['pourcentage'] = (gdf_idf_ophtalmo_1500m['nbre_patients_<=1500m']/gdf_idf_ophtalmo_1500m['nb_patients_ophtalmo']*100).round(1)

print(gdf_idf_hemato_1500m['pourcentage'])
print(gdf_idf_sein_1500m['pourcentage'])
print(gdf_idf_uro_1500m['pourcentage'])
print(gdf_idf_ophtalmo_1500m['pourcentage'])

0    99.7
1    67.8
2    72.3
3    74.3
4    92.1
5    84.6
6    83.9
7    78.9
Name: pourcentage, dtype: float64
0    99.9
1    68.8
2    72.8
3    75.1
4    92.3
5    84.9
6    89.0
7    85.2
Name: pourcentage, dtype: float64
0    100.0
1     68.4
2     73.4
3     77.6
4     93.8
5     85.5
6     85.2
7     83.9
Name: pourcentage, dtype: float64
0    100.0
1     65.0
2     71.0
3     65.1
4     96.4
5     87.1
6     87.5
7     81.7
Name: pourcentage, dtype: float64


In [1374]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_1500m['NOM_DEPT'], y=gdf_idf_hemato_1500m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_1500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_1500m['NOM_DEPT'], y=gdf_idf_sein_1500m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_1500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_1500m['NOM_DEPT'], y=gdf_idf_uro_1500m['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_1500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_1500m['NOM_DEPT'], y=gdf_idf_ophtalmo_1500m['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_1500m['pourcentage'],
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle en fonction de la pathologie située a une distance maximale de 1.5km des voies ferrées par departement')

fig.show()

### PERIMETRE DE 2.5 KM

#### 1 - Repartition de l'ensemble de patients dans cette zone

In [1375]:
df_2500m= pd.read_csv("Resultats/Excels/patients_FR_IDF_geocoded_adultes_clinique_2.5km.csv", sep=";")
patient_counts_exp2500m_by_dept = df_2500m.groupby('CODE_DEPT_').size().reset_index(name='exposes')

df_2500m.columns

Index(['field_1', 'pseudo_pro', 'adresse', 'codepost', 'nom_commun', 'requete',
       'x', 'y', 'score', 'trust_scor', 'street', 'city', 'pc_city', 'ic_city',
       'code_dept', 'dept', 'reg', 'address', 'address_ha', 'address__1',
       'same_city', 'hostel', 'hosted', 'date_geolo', 'geometry', 'CODE_IRIS',
       'INSEE_REG', 'CODE_DEPT_', 'patient_se', 'date_naiss', 'centre',
       'ageaudiag', 'cancernum', 'date_diag', 'topo_initi', 'topo_ini_1'],
      dtype='object')

##### Pourcentage sur l'ensemble de la region d'IDF

In [1376]:
len(df_2500m.drop_duplicates(subset='pseudo_pro'))

42727

In [1377]:
pourcentage = len(df_2500m.drop_duplicates(subset='pseudo_pro'))/len(df_patients.drop_duplicates(subset='pseudo_provisoire'))*100
pourcentage.__round__(2)

95.43

##### Répartition en fonction de chaque département

In [1378]:
patient_counts_exp2500m_by_dept['CODE_DEPT_'] = patient_counts_exp2500m_by_dept['CODE_DEPT_'].astype(str)

In [1379]:
stats_2500m = patients_counts_by_dept.merge(patient_counts_exp2500m_by_dept, left_on='CODE_DEPT', right_on='CODE_DEPT_')
stats_2500m.drop(columns=['CODE_DEPT_'], inplace=True)
stats_2500m

,CODE_DEPT,NOM_DEPT,patient_co,exposes
0,75,PARIS,10892,10892
1,77,SEINE-ET-MARNE,3018,2512
2,78,YVELINES,8628,7736
3,91,ESSONNE,2992,2689
4,92,HAUTS-DE-SEINE,10174,10151
5,93,SEINE-SAINT-DENIS,3204,3135
6,94,VAL-DE-MARNE,3038,2949
7,95,VAL-D'OISE,2824,2663


In [1380]:
stats_2500m['pourcentage_2500'] = (stats_2500m['exposes'] / stats_2500m['patient_co'])*100
stats_2500m = stats_2500m.sort_values(by ='CODE_DEPT')
stats_2500m

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_2500
0,75,PARIS,10892,10892,100.000000
1,77,SEINE-ET-MARNE,3018,2512,83.233930
2,78,YVELINES,8628,7736,89.661567
3,91,ESSONNE,2992,2689,89.872995
4,92,HAUTS-DE-SEINE,10174,10151,99.773934
5,93,SEINE-SAINT-DENIS,3204,3135,97.846442
6,94,VAL-DE-MARNE,3038,2949,97.070441
7,95,VAL-D'OISE,2824,2663,94.298867


In [1381]:
fig = px.bar(stats_2500m, x='NOM_DEPT', y='pourcentage_2500', title='Patientèle située a une distance maximale de 2.5km des voies ferrées par departement',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_2500': 'patients (%)'})
fig.show()

In [1382]:
gdf_buffer_2_5km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/voies_ferrees_idf/zone_2.5_km.shp')
gdf_buffer_2_5km = gdf_buffer_2_5km.to_crs(gdf_patients_patho_idf.crs)

In [1383]:
gdf_idf_hemato_2500m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_hemato_2500m_dept = gdf_idf_hemato_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_hemato_2500m = gdf_idf_hemato_2500m_dept .merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_2500m['pourcentage']= (gdf_idf_hemato_2500m['nbre_patients_<=2500m']/gdf_idf_hemato_2500m['nb_patients_hemato']*100).round(1)



gdf_idf_sein_2500m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_sein_2500m_dept = gdf_idf_sein_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_sein_2500m = gdf_idf_sein_2500m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_2500m['pourcentage']= (gdf_idf_sein_2500m['nbre_patients_<=2500m']/gdf_idf_sein_2500m['nb_patients_sein']*100).round(1)


gdf_idf_uro_2500m = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_uro_2500m_dept = gdf_idf_uro_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_uro_2500m = gdf_idf_uro_2500m_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_2500m['pourcentage']= (gdf_idf_uro_2500m['nbre_patients_<=2500m']/gdf_idf_uro_2500m['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_2500m = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_ophtalmo_2500m_dept = gdf_idf_ophtalmo_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_ophtalmo_2500m = gdf_idf_ophtalmo_2500m_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_2500m['pourcentage'] = (gdf_idf_ophtalmo_2500m['nbre_patients_<=2500m']/gdf_idf_ophtalmo_2500m['nb_patients_ophtalmo']*100).round(1)

gdf_idf_thorax_2500m = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_thorax_2500m_dept = gdf_idf_thorax_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_thorax_2500m = gdf_idf_thorax_2500m_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_2500m['pourcentage'] = (gdf_idf_thorax_2500m['nbre_patients_<=2500m']/gdf_idf_thorax_2500m['nb_patients_thorax']*100).round(1)


gdf_idf_gastro_2500m = gpd.sjoin(gdf_idf_gastro_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_gastro_2500m_dept = gdf_idf_gastro_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_gastro_2500m  = gdf_idf_gastro_2500m_dept.merge(gdf_idf_gastro_dept, on='CODE_DEPT')
gdf_idf_gastro_2500m ['pourcentage'] = (gdf_idf_gastro_2500m['nbre_patients_<=2500m']/gdf_idf_gastro_2500m['nb_patients_gastro']*100).round(1)

gdf_idf_gyneco_2500m  = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_2_5km , how='inner', predicate='intersects')
gdf_idf_gyneco_2500m_dept = gdf_idf_gyneco_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_gyneco_2500m = gdf_idf_gyneco_2500m_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_2500m['pourcentage'] = (gdf_idf_gyneco_2500m['nbre_patients_<=2500m']/gdf_idf_gyneco_2500m['nb_patients_gyneco']*100).round(1)

gdf_idf_sarcome_2500m = gpd.sjoin(gdf_idf_sarcome_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_sarcome_2500m_dept = gdf_idf_sarcome_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_sarcome_2500m = gdf_idf_sarcome_2500m_dept.merge(gdf_idf_sarcome_dept, on='CODE_DEPT')
gdf_idf_sarcome_2500m['pourcentage'] = (gdf_idf_sarcome_2500m['nbre_patients_<=2500m']/gdf_idf_sarcome_2500m['nb_patients_sarcome']*100).round(1)

gdf_idf_orl_2500m = gpd.sjoin(gdf_idf_orl_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_orl_2500m_dept = gdf_idf_orl_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_orl_2500m = gdf_idf_orl_2500m_dept.merge(gdf_idf_orl_dept, on='CODE_DEPT')
gdf_idf_orl_2500m['pourcentage'] = (gdf_idf_orl_2500m['nbre_patients_<=2500m']/gdf_idf_orl_2500m['nb_patients_orl']*100).round(1)

gdf_idf_dermato_2500m = gpd.sjoin(gdf_idf_dermato_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_dermato_2500m_dept = gdf_idf_dermato_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_dermato_2500m= gdf_idf_dermato_2500m_dept.merge(gdf_idf_dermato_dept, on='CODE_DEPT')
gdf_idf_dermato_2500m['pourcentage'] = (gdf_idf_dermato_2500m['nbre_patients_<=2500m']/gdf_idf_dermato_2500m['nb_patients_dermato']*100).round(1)

gdf_idf_endocrino_2500m = gpd.sjoin(gdf_idf_endocrino_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_endocrino_2500m_dept = gdf_idf_endocrino_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_endocrino_2500m = gdf_idf_endocrino_2500m_dept.merge(gdf_idf_endocrino_dept, on='CODE_DEPT')
gdf_idf_endocrino_2500m['pourcentage'] = (gdf_idf_endocrino_2500m['nbre_patients_<=2500m']/gdf_idf_endocrino_2500m['nb_patients_endocrino']*100).round(1)

gdf_idf_neuro_2500m = gpd.sjoin(gdf_idf_neuro_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_neuro_2500m_dept = gdf_idf_neuro_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_neuro_2500m  = gdf_idf_neuro_2500m_dept.merge(gdf_idf_neuro_dept, on='CODE_DEPT')
gdf_idf_neuro_2500m['pourcentage'] = (gdf_idf_neuro_2500m['nbre_patients_<=2500m']/gdf_idf_neuro_2500m['nb_patients_neuro']*100).round(1)

gdf_idf_autre_2500m = gpd.sjoin(gdf_idf_autre_points, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_idf_autre_2500m_dept = gdf_idf_autre_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_autre_2500m = gdf_idf_autre_2500m_dept.merge(gdf_idf_autre_dept, on='CODE_DEPT')
gdf_idf_autre_2500m['pourcentage'] = (gdf_idf_autre_2500m['nbre_patients_<=2500m']/gdf_idf_autre_2500m['nb_patients_autre']*100).round(1)


print(gdf_idf_hemato_2500m['pourcentage'])
print(gdf_idf_sein_2500m['pourcentage'])
print(gdf_idf_uro_2500m['pourcentage'])
print(gdf_idf_ophtalmo_2500m['pourcentage'])

0    100.0
1     82.2
2     91.2
3     89.1
4     99.7
5    100.0
6     93.5
7     95.8
Name: pourcentage, dtype: float64
0    100.0
1     83.6
2     89.2
3     90.4
4     99.8
5     97.6
6     97.2
7     94.2
Name: pourcentage, dtype: float64
0    100.0
1     86.0
2     90.3
3     90.3
4     99.9
5     98.2
6     97.2
7     93.3
Name: pourcentage, dtype: float64
0    100.0
1     79.0
2     87.9
3     85.5
4    100.0
5     98.9
6     95.8
7     97.8
Name: pourcentage, dtype: float64


In [1384]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_2500m['NOM_DEPT'], y=gdf_idf_hemato_2500m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_2500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_2500m['NOM_DEPT'], y=gdf_idf_sein_2500m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_2500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_2500m['NOM_DEPT'], y=gdf_idf_uro_2500m['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_2500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_2500m['NOM_DEPT'], y=gdf_idf_ophtalmo_2500m['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_2500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_thorax_2500m['NOM_DEPT'], y=gdf_idf_thorax_2500m['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_2500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle en fonction de la pathologie située a une distance maximale de 2.5km des voies ferrées par departement')

fig.show()

In [1385]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_gastro_2500m['NOM_DEPT'], y=gdf_idf_gastro_2500m['pourcentage'], name='cancer gastro',
                     text=gdf_idf_gastro_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_2500m['NOM_DEPT'], y=gdf_idf_gyneco_2500m['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sarcome_2500m['NOM_DEPT'], y=gdf_idf_sarcome_2500m['pourcentage'], name='cancer sarcome',
                     text=gdf_idf_sarcome_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_orl_2500m['NOM_DEPT'], y=gdf_idf_orl_2500m['pourcentage'], name='cancer orl',
                     text=gdf_idf_orl_2500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 2500m des voies ferrées en fonction de leur groupe pathologique')

fig.show()

In [1386]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_dermato_2500m['NOM_DEPT'], y=gdf_idf_dermato_2500m['pourcentage'], name='cancer dermato',
                     text=gdf_idf_dermato_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_endocrino_2500m['NOM_DEPT'], y=gdf_idf_endocrino_2500m['pourcentage'], name='cancer endocrino',
                     text=gdf_idf_endocrino_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_neuro_2500m['NOM_DEPT'], y=gdf_idf_neuro_2500m['pourcentage'], name='cancer neuro',
                     text=gdf_idf_neuro_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_autre_2500m['NOM_DEPT'], y=gdf_idf_autre_2500m['pourcentage'], name='cancer autre',
                     text=gdf_idf_autre_2500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 2500m des voies ferrées en fonction de leur groupe pathologique')

fig.show()

## TABLEAU RECAPITULATIF DE L'ENSEMBLE DE LA PATIENTELE EN FONCTION DE LEUR PROXIMITE AUX VOIES FERREES

In [1387]:
# Création d'une figure avec des sous-trames
fig = fig = go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=stats_150m['NOM_DEPT'], y=stats_150m['pourcentage_150'], name='périmètre 150m',
                     text=stats_150m['pourcentage_150'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=stats_500m['NOM_DEPT'], y=stats_500m['pourcentage_500'], name='périmètre 500m',
                     text=stats_500m['pourcentage_500'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=stats_1000m['NOM_DEPT'], y=stats_1000m['pourcentage_1000'], name='périmètre 1000m',
                     text=stats_1000m['pourcentage_1000'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=stats_1500m['NOM_DEPT'], y=stats_1500m['pourcentage_1500'], name='périmètre 1500m',
                     text=stats_1500m['pourcentage_1500'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=stats_2500m['NOM_DEPT'], y=stats_2500m['pourcentage_2500'], name='périmètre 2500m',
                     text=stats_2500m['pourcentage_2500'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle par département en fonction de leur proximité aux voies ferrees')

fig.show()

In [1388]:
# # Création d'une figure avec des sous-trames
# fig = fig = go.Figure()

# # Ajout des bar plots pour chaque périmètre
# fig.add_trace(go.Bar(x=stats_150m['NOM_DEPT'], y=stats_150m['pourcentage_150'], name='périmètre 150m',
#                      text=stats_150m['pourcentage_150'].round(1),
#                      textposition='outside'))
# fig.add_trace(go.Bar(x=stats_500m['NOM_DEPT'], y=stats_500m['pourcentage_500'], name='périmètre 500m',
#                      text=stats_500m['pourcentage_500'].round(1),
#                      textposition='outside'))
# fig.add_trace(go.Bar(x=stats_1000m['NOM_DEPT'], y=stats_1000m['pourcentage_1000'], name='périmètre 1000m',
#                      text=stats_1000m['pourcentage_1000'].round(1),
#                      textposition='outside'))
# fig.add_trace(go.Bar(x=stats_2500m['NOM_DEPT'], y=stats_2500m['pourcentage_2500'], name='périmètre 2500m',
#                      text=stats_2500m['pourcentage_2500'].round(1),
#                      textposition='outside'))


# fig.update_layout(barmode='group', 
#                   xaxis=dict(title='Département'),
#                   yaxis=dict(title='% Patients'),
#                   title='Patientèle par département en fonction de leur proximité aux voies ferrees')

# fig.show()

LES PATIENTS DE PARIS DE PATHOLOGIE THORAX

...

# STATISTIQUES EN FONCTION DE LA PROXIMITE AUX VOIES ROUTIERES EN IDF

In [1389]:
gdf_buffer_150m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_150m.shp')
gdf_buffer_500m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_500m.shp')
gdf_buffer_1km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_1km.shp')
gdf_buffer_1_5km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_1_5km.shp')
gdf_buffer_2_5km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_2_5km.shp')

In [1390]:
gdf_patients_patho_idf = gdf_patients_patho_idf.to_crs(gdf_dept.crs)
gdf_patients_patho_idf.columns

Index(['Unnamed: 0', 'pseudo_provisoire', 'adresse', 'codepost',
       'nom_commune_postal', 'requete', 'x', 'y', 'score', 'trust_score',
       'street', 'city', 'pc_city', 'ic_city', 'code_dept', 'dept', 'reg',
       'address', 'address_has_num_init', 'address_has_num_geoloc',
       'same_city', 'hostel', 'hosted', 'date_geoloc', 'geometry', 'CODE_IRIS',
       'INSEE_REG', 'CODE_DEPT', 'patient_sexe', 'date_naissance', 'centre',
       'ageaudiag', 'cancernum', 'date_diag', 'topo_initiale_cim10',
       'topo_initialelib', 'patho', 'patient_co'],
      dtype='object')

In [1391]:
columns_keep = gdf_patients_patho_idf.drop(columns = ['Unnamed: 0']).columns

In [1392]:

gdf_150m = gpd.sjoin(gdf_patients_patho_idf, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_150m = gdf_150m[columns_keep]

gdf_500m = gpd.sjoin(gdf_patients_patho_idf, gdf_buffer_500m, how='inner', predicate='intersects')
gdf_500m = gdf_500m[columns_keep]

gdf_1km = gpd.sjoin(gdf_patients_patho_idf, gdf_buffer_1km, how='inner', predicate='intersects')
gdf_1km = gdf_1km[columns_keep]

gdf_1_5km = gpd.sjoin(gdf_patients_patho_idf, gdf_buffer_1_5km, how='inner', predicate='intersects')
gdf_1_5km = gdf_1_5km[columns_keep]

gdf_2_5km = gpd.sjoin(gdf_patients_patho_idf, gdf_buffer_2_5km, how='inner', predicate='intersects')
gdf_2_5km = gdf_2_5km[columns_keep]



In [1393]:
gdf_150m.to_csv("Resultats/Excels/patients_highways/patients_FR_IDF_geocoded_adultes_clinique_150m.csv",sep=";")
gdf_500m.to_csv("Resultats/Excels/patients_highways/patients_FR_IDF_geocoded_adultes_clinique_500m.csv",sep=";")
gdf_1km.to_csv("Resultats/Excels/patients_highways/patients_FR_IDF_geocoded_adultes_clinique_1km.csv",sep=";")
gdf_1_5km.to_csv("Resultats/Excels/patients_highways/patients_FR_IDF_geocoded_adultes_clinique_1_5km.csv",sep=";")
gdf_2_5km.to_csv("Resultats/Excels/patients_highways/patients_FR_IDF_geocoded_adultes_clinique_2_5km.csv",sep=";")

In [1394]:
gdf_2_5km.columns

Index(['pseudo_provisoire', 'adresse', 'codepost', 'nom_commune_postal',
       'requete', 'x', 'y', 'score', 'trust_score', 'street', 'city',
       'pc_city', 'ic_city', 'code_dept', 'dept', 'reg', 'address',
       'address_has_num_init', 'address_has_num_geoloc', 'same_city', 'hostel',
       'hosted', 'date_geoloc', 'geometry', 'CODE_IRIS', 'INSEE_REG',
       'CODE_DEPT', 'patient_sexe', 'date_naissance', 'centre', 'ageaudiag',
       'cancernum', 'date_diag', 'topo_initiale_cim10', 'topo_initialelib',
       'patho', 'patient_co'],
      dtype='object')

### DANS UN PERIMETRE DE 150M 

#### 1- Repartition de l'ensemble des patients dans cette zone

##### Pourcentage dans l'ensemble de la région d'IDF

In [1395]:
len(gdf_150m.drop_duplicates(subset='pseudo_provisoire'))

1052

In [1396]:
pourcentage = len(gdf_150m.drop_duplicates(subset='pseudo_provisoire'))/len(df_patients.drop_duplicates(subset='pseudo_provisoire'))*100
pourcentage.__round__(2)
print(f"{pourcentage:.2f}% de patients sont situés à moins de 150m des voies routières principales en IDF")

2.35% de patients sont situés à moins de 150m des voies routières principales en IDF


##### Repartition par département

In [1397]:
patients_counts_exp150_by_dept = gdf_150m.groupby('CODE_DEPT').size().reset_index(name='exposes')
patients_counts_exp150_by_dept

,CODE_DEPT,exposes
0,75,158
1,77,46
2,78,125
3,91,35
4,92,385
5,93,107
6,94,160
7,95,36


In [1398]:
patients_counts_exp150_by_dept['CODE_DEPT'] = patients_counts_exp150_by_dept['CODE_DEPT'].astype(str)

In [1399]:
stats_routes_150m = patients_counts_by_dept.merge(patients_counts_exp150_by_dept, on='CODE_DEPT')
stats_routes_150m['pourcentage_150'] = (((stats_routes_150m['exposes'] / stats_routes_150m['patient_co'])*100).round(1))
stats_routes_150m = stats_routes_150m.sort_values(by ='CODE_DEPT')
stats_routes_150m

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_150
0,75,PARIS,10892,158,1.5
1,77,SEINE-ET-MARNE,3018,46,1.5
2,78,YVELINES,8628,125,1.4
3,91,ESSONNE,2992,35,1.2
4,92,HAUTS-DE-SEINE,10174,385,3.8
5,93,SEINE-SAINT-DENIS,3204,107,3.3
6,94,VAL-DE-MARNE,3038,160,5.3
7,95,VAL-D'OISE,2824,36,1.3


In [1400]:
fig = px.bar(stats_routes_150m, x='NOM_DEPT', y='pourcentage_150', title='Patientèle située a une distance maximale de 150m des voies routières principales par departement',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_150': 'patients (%)'})
fig.show()

#### 1- Repartition des patients par pathologie dans ce périmetre

In [1401]:
gdf_idf_hemato_150m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_hemato_150m_dept = gdf_idf_hemato_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_hemato_150m = gdf_idf_hemato_150m_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_150m['pourcentage']= (gdf_idf_hemato_150m['nbre_patients_<=150m']/gdf_idf_hemato_150m['nb_patients_hemato']*100).round(1)


gdf_idf_sein_150m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_sein_150m_dept = gdf_idf_sein_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_sein_150m = gdf_idf_sein_150m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_150m['pourcentage']= (gdf_idf_sein_150m['nbre_patients_<=150m']/gdf_idf_sein_150m['nb_patients_sein']*100).round(1)


gdf_idf_uro_150m = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_uro_150m_dept = gdf_idf_uro_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_uro_150m = gdf_idf_uro_150m_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_150m['pourcentage']= (gdf_idf_uro_150m['nbre_patients_<=150m']/gdf_idf_uro_150m['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_150m = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_150m, how='inner', predicate='intersects')
gdf_idf_ophtalmo_150m_dept = gdf_idf_ophtalmo_150m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=150m')
gdf_idf_ophtalmo_150m = gdf_idf_ophtalmo_150m_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_150m['pourcentage'] = (gdf_idf_ophtalmo_150m['nbre_patients_<=150m']/gdf_idf_ophtalmo_150m['nb_patients_ophtalmo']*100).round(1)



print(gdf_idf_sein_150m)
print(gdf_idf_hemato_150m)
print(gdf_idf_uro_150m)
print(gdf_idf_ophtalmo_150m)

  CODE_DEPT  nbre_patients_<=150m  nb_patients_sein           NOM_DEPT  \
0        75                    93              6754              PARIS   
1        77                    32              2096     SEINE-ET-MARNE   
2        78                    87              5823           YVELINES   
3        91                    25              1983            ESSONNE   
4        92                   233              6129     HAUTS-DE-SEINE   
5        93                    70              2112  SEINE-SAINT-DENIS   
6        94                   101              2003       VAL-DE-MARNE   
7        95                    22              1866         VAL-D'OISE   

   pourcentage  
0          1.4  
1          1.5  
2          1.5  
3          1.3  
4          3.8  
5          3.3  
6          5.0  
7          1.2  
  CODE_DEPT  nbre_patients_<=150m  nb_patients_hemato           NOM_DEPT  \
0        75                    10                 357              PARIS   
1        77                 

In [1402]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_150m['NOM_DEPT'], y=gdf_idf_hemato_150m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_150m['NOM_DEPT'], y=gdf_idf_sein_150m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_150m['NOM_DEPT'], y=gdf_idf_uro_150m['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_150m['NOM_DEPT'], y=gdf_idf_ophtalmo_150m['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_150m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 150m des voies routières principales en fonction de leur groupe pathologique')

fig.show()

### PERIMETRE DE 500M

#### 1- Repartition de l'ensemble des patients dans cette zone

##### Pourcentage dans l'ensemble de la région d'IDF

In [1403]:
len(gdf_500m.drop_duplicates(subset='pseudo_provisoire'))

6692

In [1404]:
pourcentage = len(gdf_500m.drop_duplicates(subset='pseudo_provisoire'))/len(df_patients.drop_duplicates(subset='pseudo_provisoire'))*100
pourcentage.__round__(2)
print(f"{pourcentage:.2f}% de patients sont situés à moins de 500m des voies ferrées en IDF")

14.95% de patients sont situés à moins de 500m des voies ferrées en IDF


##### Repartition par département

In [1405]:
patients_counts_exp500_by_dept = gdf_500m.groupby('CODE_DEPT').size().reset_index(name='exposes')
patients_counts_exp500_by_dept

,CODE_DEPT,exposes
0,75,1332
1,77,269
2,78,822
3,91,275
4,92,2142
5,93,731
6,94,790
7,95,331


In [1406]:
patients_counts_exp500_by_dept['CODE_DEPT'] = patients_counts_exp500_by_dept['CODE_DEPT'].astype(str)

In [1407]:
stats_routes_500m = patients_counts_by_dept.merge(patients_counts_exp500_by_dept, on='CODE_DEPT')
stats_routes_500m['pourcentage_500'] = (((stats_routes_500m['exposes'] / stats_routes_500m['patient_co'])*100).round(1))
stats_routes_500m = stats_routes_500m.sort_values(by ='CODE_DEPT')
stats_routes_500m

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_500
0,75,PARIS,10892,1332,12.2
1,77,SEINE-ET-MARNE,3018,269,8.9
2,78,YVELINES,8628,822,9.5
3,91,ESSONNE,2992,275,9.2
4,92,HAUTS-DE-SEINE,10174,2142,21.1
5,93,SEINE-SAINT-DENIS,3204,731,22.8
6,94,VAL-DE-MARNE,3038,790,26.0
7,95,VAL-D'OISE,2824,331,11.7


In [1408]:
fig = px.bar(stats_routes_500m, x='NOM_DEPT', y='pourcentage_500', title='Patientèle située a une distance maximale de 500m des voies routières principales par departement',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_500': 'patients (%)'})
fig.show()

#### 2- Repartition par pathologie cette zone

In [1409]:
gdf_buffer_routes_500m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_500m.shp')
gdf_buffer_routes_500m = gdf_buffer_routes_500m.to_crs(gdf_patients_patho_idf.crs)

In [1410]:
gdf_idf_hemato_500m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_hemato_500m_dept = gdf_idf_hemato_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_hemato_500m = gdf_idf_hemato_500m_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_500m['pourcentage']= (gdf_idf_hemato_500m['nbre_patients_<=500m']/gdf_idf_hemato_500m['nb_patients_hemato']*100).round(1)



gdf_idf_sein_500m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_sein_500m_dept = gdf_idf_sein_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_sein_500m = gdf_idf_sein_500m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_500m['pourcentage']= (gdf_idf_sein_500m['nbre_patients_<=500m']/gdf_idf_sein_500m['nb_patients_sein']*100).round(1)


gdf_idf_uro_500m = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_uro_500m_dept = gdf_idf_uro_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_uro_500m = gdf_idf_uro_500m_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_500m['pourcentage']= (gdf_idf_uro_500m['nbre_patients_<=500m']/gdf_idf_uro_150m['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_500m = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_ophtalmo_500m_dept = gdf_idf_ophtalmo_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_ophtalmo_500m = gdf_idf_ophtalmo_500m_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_500m['pourcentage'] = (gdf_idf_ophtalmo_500m['nbre_patients_<=500m']/gdf_idf_ophtalmo_500m['nb_patients_ophtalmo']*100).round(1)


gdf_idf_thorax_500m = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_thorax_500m_dept = gdf_idf_thorax_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_thorax_500m = gdf_idf_thorax_500m_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_500m['pourcentage'] = (gdf_idf_thorax_500m['nbre_patients_<=500m']/gdf_idf_thorax_500m['nb_patients_thorax']*100).round(1)

gdf_idf_gastro_500m = gpd.sjoin(gdf_idf_gastro_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_gastro_500m_dept = gdf_idf_gastro_500m .groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_gastro_500m  = gdf_idf_gastro_500m_dept.merge(gdf_idf_gastro_dept, on='CODE_DEPT')
gdf_idf_gastro_500m ['pourcentage'] = (gdf_idf_gastro_500m['nbre_patients_<=500m']/gdf_idf_gastro_500m['nb_patients_gastro']*100).round(1)

gdf_idf_gyneco_500m  = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_routes_500m , how='inner', predicate='intersects')
gdf_idf_gyneco_500m_dept = gdf_idf_gyneco_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_gyneco_500m = gdf_idf_gyneco_500m_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_500m['pourcentage'] = (gdf_idf_gyneco_500m['nbre_patients_<=500m']/gdf_idf_gyneco_500m['nb_patients_gyneco']*100).round(1)

gdf_idf_sarcome_500m = gpd.sjoin(gdf_idf_sarcome_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_sarcome_500m_dept = gdf_idf_sarcome_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_sarcome_500m  = gdf_idf_sarcome_500m_dept.merge(gdf_idf_sarcome_dept, on='CODE_DEPT')
gdf_idf_sarcome_500m['pourcentage'] = (gdf_idf_sarcome_500m['nbre_patients_<=500m']/gdf_idf_sarcome_500m['nb_patients_sarcome']*100).round(1)

gdf_idf_orl_500m = gpd.sjoin(gdf_idf_orl_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_orl_500m_dept = gdf_idf_orl_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_orl_500m = gdf_idf_orl_500m_dept.merge(gdf_idf_orl_dept, on='CODE_DEPT')
gdf_idf_orl_500m['pourcentage'] = (gdf_idf_orl_500m['nbre_patients_<=500m']/gdf_idf_orl_500m['nb_patients_orl']*100).round(1)

gdf_idf_dermato_500m = gpd.sjoin(gdf_idf_dermato_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_dermato_500m_dept = gdf_idf_dermato_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_dermato_500m= gdf_idf_dermato_500m_dept.merge(gdf_idf_dermato_dept, on='CODE_DEPT')
gdf_idf_dermato_500m['pourcentage'] = (gdf_idf_dermato_500m['nbre_patients_<=500m']/gdf_idf_dermato_500m['nb_patients_dermato']*100).round(1)

gdf_idf_endocrino_500m = gpd.sjoin(gdf_idf_endocrino_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_endocrino_500m_dept = gdf_idf_endocrino_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_endocrino_500m = gdf_idf_endocrino_500m_dept.merge(gdf_idf_endocrino_dept, on='CODE_DEPT')
gdf_idf_endocrino_500m['pourcentage'] = (gdf_idf_endocrino_500m['nbre_patients_<=500m']/gdf_idf_endocrino_500m['nb_patients_endocrino']*100).round(1)

gdf_idf_neuro_500m = gpd.sjoin(gdf_idf_neuro_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_neuro_500m_dept = gdf_idf_neuro_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_neuro_500m  = gdf_idf_neuro_500m_dept.merge(gdf_idf_neuro_dept, on='CODE_DEPT')
gdf_idf_neuro_500m['pourcentage'] = (gdf_idf_neuro_500m['nbre_patients_<=500m']/gdf_idf_neuro_500m['nb_patients_neuro']*100).round(1)

gdf_idf_autre_500m = gpd.sjoin(gdf_idf_autre_points, gdf_buffer_routes_500m, how='inner', predicate='intersects')
gdf_idf_autre_500m_dept = gdf_idf_autre_500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=500m')
gdf_idf_autre_500m = gdf_idf_autre_500m_dept.merge(gdf_idf_autre_dept, on='CODE_DEPT')
gdf_idf_autre_500m['pourcentage'] = (gdf_idf_autre_500m['nbre_patients_<=500m']/gdf_idf_autre_500m['nb_patients_autre']*100).round(1)


print(gdf_idf_sein_500m)
print(gdf_idf_hemato_500m)
print(gdf_idf_uro_500m)
print(gdf_idf_ophtalmo_500m)

  CODE_DEPT  nbre_patients_<=500m  nb_patients_sein           NOM_DEPT  \
0        75                   808              6754              PARIS   
1        77                   187              2096     SEINE-ET-MARNE   
2        78                   562              5823           YVELINES   
3        91                   183              1983            ESSONNE   
4        92                  1266              6129     HAUTS-DE-SEINE   
5        93                   477              2112  SEINE-SAINT-DENIS   
6        94                   532              2003       VAL-DE-MARNE   
7        95                   232              1866         VAL-D'OISE   

   pourcentage  
0         12.0  
1          8.9  
2          9.7  
3          9.2  
4         20.7  
5         22.6  
6         26.6  
7         12.4  
  CODE_DEPT  nbre_patients_<=500m  nb_patients_hemato           NOM_DEPT  \
0        75                    65                 357              PARIS   
1        77                 

In [1411]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_500m['NOM_DEPT'], y=gdf_idf_hemato_500m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_500m['NOM_DEPT'], y=gdf_idf_sein_500m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_500m['NOM_DEPT'], y=gdf_idf_uro_500m['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_500m['NOM_DEPT'], y=gdf_idf_ophtalmo_500m['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_thorax_500m['NOM_DEPT'], y=gdf_idf_thorax_500m['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située a une distance maximale de 500m des routes principales par departement en fonction de leur groupe pathologique')

fig.show()

In [1455]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_gastro_500m['NOM_DEPT'], y=gdf_idf_gastro_500m['pourcentage'], name='cancer gastro',
                     text=gdf_idf_gastro_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_500m['NOM_DEPT'], y=gdf_idf_gyneco_500m['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_150m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sarcome_500m['NOM_DEPT'], y=gdf_idf_sarcome_500m['pourcentage'], name='cancer sarcome',
                     text=gdf_idf_sarcome_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_orl_500m['NOM_DEPT'], y=gdf_idf_orl_500m['pourcentage'], name='cancer orl',
                     text=gdf_idf_orl_500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 500m des autoroutes en fonction de leur groupe pathologique')

fig.show()

In [1453]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_dermato_500m['NOM_DEPT'], y=gdf_idf_dermato_500m['pourcentage'], name='cancer dermato',
                     text=gdf_idf_dermato_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_endocrino_500m['NOM_DEPT'], y=gdf_idf_endocrino_500m['pourcentage'], name='cancer endocrino',
                     text=gdf_idf_endocrino_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_neuro_500m['NOM_DEPT'], y=gdf_idf_neuro_500m['pourcentage'], name='cancer neuro',
                     text=gdf_idf_neuro_500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_autre_500m['NOM_DEPT'], y=gdf_idf_autre_500m['pourcentage'], name='cancer autre',
                     text=gdf_idf_autre_500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 500m des routes principales en fonction de leur groupe pathologique')

fig.show()

### PERIMETRE DE 1KM

#### 1- Repartition de l'ensemble des patients dans cette zone

##### Pourcentage dans l'ensemble de la région d'IDF

In [1414]:
len(gdf_1km.drop_duplicates(subset='pseudo_provisoire'))

15918

In [1415]:
pourcentage = len(gdf_1km.drop_duplicates(subset='pseudo_provisoire'))/len(df_patients.drop_duplicates(subset='pseudo_provisoire'))*100
pourcentage.__round__(2)
print(f"{pourcentage:.2f}% de patients sont situés à moins d\'un km des voies routieres principales en IDF")

35.55% de patients sont situés à moins d'un km des voies routieres principales en IDF


##### Repartition par departement

In [1416]:
patients_counts_exp1km_by_dept = gdf_1km.groupby('CODE_DEPT').size().reset_index(name='exposes')
patients_counts_exp1km_by_dept

,CODE_DEPT,exposes
0,75,3739
1,77,628
2,78,2206
3,91,668
4,92,4799
5,93,1494
6,94,1483
7,95,901


In [1417]:
patients_counts_exp1km_by_dept['CODE_DEPT'] = patients_counts_exp1km_by_dept['CODE_DEPT'].astype(str)

In [1418]:
stats_routes_1km = patients_counts_by_dept.merge(patients_counts_exp1km_by_dept, on='CODE_DEPT')
stats_routes_1km['pourcentage_1km'] = (((stats_routes_1km['exposes'] / stats_routes_1km['patient_co'])*100).round(1))
stats_routes_1km = stats_routes_1km.sort_values(by ='CODE_DEPT')
stats_routes_1km

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_1km
0,75,PARIS,10892,3739,34.3
1,77,SEINE-ET-MARNE,3018,628,20.8
2,78,YVELINES,8628,2206,25.6
3,91,ESSONNE,2992,668,22.3
4,92,HAUTS-DE-SEINE,10174,4799,47.2
5,93,SEINE-SAINT-DENIS,3204,1494,46.6
6,94,VAL-DE-MARNE,3038,1483,48.8
7,95,VAL-D'OISE,2824,901,31.9


In [1419]:
fig = px.bar(stats_routes_1km, x='NOM_DEPT', y='pourcentage_1km', title='Patientèle située a une distance maximale d\'un km des voies routières principales par departement',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_1km': 'patients (%)'})
fig.show()

#### 2- Repartition par pathologie dans cette zone

In [1420]:
gdf_buffer_routes_1km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_1km.shp')
gdf_buffer_routes_1km = gdf_buffer_routes_1km.to_crs(gdf_patients_patho_idf.crs)

In [1421]:
gdf_idf_hemato_1km = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_hemato_1km_dept = gdf_idf_hemato_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_hemato_1km = gdf_idf_hemato_1km_dept.merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_1km['pourcentage']= (gdf_idf_hemato_1km['nbre_patients_<=1km']/gdf_idf_hemato_1km['nb_patients_hemato']*100).round(1)


gdf_idf_sein_1km = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_sein_1km_dept = gdf_idf_sein_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_sein_1km = gdf_idf_sein_1km_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_1km['pourcentage']= (gdf_idf_sein_1km['nbre_patients_<=1km']/gdf_idf_sein_1km['nb_patients_sein']*100).round(1)


gdf_idf_uro_1km = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_uro_1km_dept = gdf_idf_uro_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_uro_1km = gdf_idf_uro_1km_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_1km['pourcentage']= (gdf_idf_uro_1km['nbre_patients_<=1km']/gdf_idf_uro_1km['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_1km = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_ophtalmo_1km_dept = gdf_idf_ophtalmo_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_ophtalmo_1km = gdf_idf_ophtalmo_1km_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_1km['pourcentage'] = (gdf_idf_ophtalmo_1km['nbre_patients_<=1km']/gdf_idf_ophtalmo_1km['nb_patients_ophtalmo']*100).round(1)

gdf_idf_thorax_1km = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_thorax_1km_dept = gdf_idf_thorax_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_thorax_1km = gdf_idf_thorax_1km_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_1km['pourcentage'] = (gdf_idf_thorax_1km['nbre_patients_<=1km']/gdf_idf_thorax_500m['nb_patients_thorax']*100).round(1)


gdf_idf_gastro_1km = gpd.sjoin(gdf_idf_gastro_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_gastro_1km_dept = gdf_idf_gastro_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_gastro_1km  = gdf_idf_gastro_1km_dept.merge(gdf_idf_gastro_dept, on='CODE_DEPT')
gdf_idf_gastro_1km ['pourcentage'] = (gdf_idf_gastro_1km['nbre_patients_<=1km']/gdf_idf_gastro_1km['nb_patients_gastro']*100).round(1)

gdf_idf_gyneco_1km  = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_routes_1km , how='inner', predicate='intersects')
gdf_idf_gyneco_1km_dept = gdf_idf_gyneco_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_gyneco_1km = gdf_idf_gyneco_1km_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_1km['pourcentage'] = (gdf_idf_gyneco_1km['nbre_patients_<=1km']/gdf_idf_gyneco_1km['nb_patients_gyneco']*100).round(1)

gdf_idf_sarcome_1km = gpd.sjoin(gdf_idf_sarcome_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_sarcome_1km_dept = gdf_idf_sarcome_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_sarcome_1km = gdf_idf_sarcome_1km_dept.merge(gdf_idf_sarcome_dept, on='CODE_DEPT')
gdf_idf_sarcome_1km['pourcentage'] = (gdf_idf_sarcome_1km['nbre_patients_<=1km']/gdf_idf_sarcome_1km['nb_patients_sarcome']*100).round(1)

gdf_idf_orl_1km = gpd.sjoin(gdf_idf_orl_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_orl_1km_dept = gdf_idf_orl_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_orl_1km = gdf_idf_orl_1km_dept.merge(gdf_idf_orl_dept, on='CODE_DEPT')
gdf_idf_orl_1km['pourcentage'] = (gdf_idf_orl_1km['nbre_patients_<=1km']/gdf_idf_orl_1km['nb_patients_orl']*100).round(1)

gdf_idf_dermato_1km = gpd.sjoin(gdf_idf_dermato_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_dermato_1km_dept = gdf_idf_dermato_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_dermato_1km= gdf_idf_dermato_1km_dept.merge(gdf_idf_dermato_dept, on='CODE_DEPT')
gdf_idf_dermato_1km['pourcentage'] = (gdf_idf_dermato_1km['nbre_patients_<=1km']/gdf_idf_dermato_1km['nb_patients_dermato']*100).round(1)

gdf_idf_endocrino_1km = gpd.sjoin(gdf_idf_endocrino_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_endocrino_1km_dept = gdf_idf_endocrino_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_endocrino_1km = gdf_idf_endocrino_1km_dept.merge(gdf_idf_endocrino_dept, on='CODE_DEPT')
gdf_idf_endocrino_1km['pourcentage'] = (gdf_idf_endocrino_1km['nbre_patients_<=1km']/gdf_idf_endocrino_1km['nb_patients_endocrino']*100).round(1)

gdf_idf_neuro_1km = gpd.sjoin(gdf_idf_neuro_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_neuro_1km_dept = gdf_idf_neuro_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_neuro_1km  = gdf_idf_neuro_1km_dept.merge(gdf_idf_neuro_dept, on='CODE_DEPT')
gdf_idf_neuro_1km['pourcentage'] = (gdf_idf_neuro_1km['nbre_patients_<=1km']/gdf_idf_neuro_1km['nb_patients_neuro']*100).round(1)

gdf_idf_autre_1km = gpd.sjoin(gdf_idf_autre_points, gdf_buffer_routes_1km, how='inner', predicate='intersects')
gdf_idf_autre_1km_dept = gdf_idf_autre_1km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1km')
gdf_idf_autre_1km = gdf_idf_autre_1km_dept.merge(gdf_idf_autre_dept, on='CODE_DEPT')
gdf_idf_autre_1km['pourcentage'] = (gdf_idf_autre_1km['nbre_patients_<=1km']/gdf_idf_autre_1km['nb_patients_autre']*100).round(1)
print(gdf_idf_sein_1km)
print(gdf_idf_hemato_1km)
print(gdf_idf_uro_1km)
print(gdf_idf_ophtalmo_1km)

  CODE_DEPT  nbre_patients_<=1km  nb_patients_sein           NOM_DEPT  \
0        75                 2306              6754              PARIS   
1        77                  429              2096     SEINE-ET-MARNE   
2        78                 1490              5823           YVELINES   
3        91                  439              1983            ESSONNE   
4        92                 2844              6129     HAUTS-DE-SEINE   
5        93                  975              2112  SEINE-SAINT-DENIS   
6        94                  983              2003       VAL-DE-MARNE   
7        95                  606              1866         VAL-D'OISE   

   pourcentage  
0         34.1  
1         20.5  
2         25.6  
3         22.1  
4         46.4  
5         46.2  
6         49.1  
7         32.5  
  CODE_DEPT  nbre_patients_<=1km  nb_patients_hemato           NOM_DEPT  \
0        75                  144                 357              PARIS   
1        77                   17       

In [1422]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_1km['NOM_DEPT'], y=gdf_idf_hemato_1km['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_1km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_1km['NOM_DEPT'], y=gdf_idf_sein_1km['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_1km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_1km['NOM_DEPT'], y=gdf_idf_uro_1km['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_1km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_1km['NOM_DEPT'], y=gdf_idf_ophtalmo_1km['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_1km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_thorax_1km['NOM_DEPT'], y=gdf_idf_thorax_1km['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_1km['pourcentage'].round(1),
                     textposition='outside'))

fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située a une distance maximale d\'un km des routes principales par departement en fonction de leur groupe pathologique')

fig.show()

In [1454]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_gastro_1km['NOM_DEPT'], y=gdf_idf_gastro_1km['pourcentage'], name='cancer gastro',
                     text=gdf_idf_gastro_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_1km['NOM_DEPT'], y=gdf_idf_gyneco_1km['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sarcome_1km['NOM_DEPT'], y=gdf_idf_sarcome_1km['pourcentage'], name='cancer sarcome',
                     text=gdf_idf_sarcome_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_orl_1km['NOM_DEPT'], y=gdf_idf_orl_1km['pourcentage'], name='cancer orl',
                     text=gdf_idf_orl_1km['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 1km des voies routières principales en fonction de leur groupe pathologique')

fig.show()

In [1452]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_dermato_1km['NOM_DEPT'], y=gdf_idf_dermato_1km['pourcentage'], name='cancer dermato',
                     text=gdf_idf_dermato_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_endocrino_1km['NOM_DEPT'], y=gdf_idf_endocrino_1km['pourcentage'], name='cancer endocrino',
                     text=gdf_idf_endocrino_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_neuro_1km['NOM_DEPT'], y=gdf_idf_neuro_1km['pourcentage'], name='cancer neuro',
                     text=gdf_idf_neuro_1km['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_autre_1km['NOM_DEPT'], y=gdf_idf_autre_1km['pourcentage'], name='cancer autre',
                     text=gdf_idf_autre_1km['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Département'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par départment et située à une distance maximale de 1km des voies routières principales en fonction de leur groupe pathologique')

fig.show()

### PERIMETRE DE 1.5KM

#### 1- Repartition de l'ensemble des patients dans cette zone

##### Pourcentage dans l'ensemble de la zone IDF

In [1425]:
len(gdf_1_5km.drop_duplicates(subset='pseudo_provisoire'))

23633

In [1426]:
pourcentage = len(gdf_1_5km.drop_duplicates(subset='pseudo_provisoire'))/len(df_patients.drop_duplicates(subset='pseudo_provisoire'))*100
pourcentage.__round__(2)
print(f"{pourcentage:.2f}% de patients sont situés à moins d\'1.5 km des voies routieres principales en IDF")

52.79% de patients sont situés à moins d'1.5 km des voies routieres principales en IDF


In [1427]:
gdf_buffer_routes_1_5km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_1_5km.shp')
gdf_buffer_routes_1_5km = gdf_buffer_routes_1_5km.to_crs(gdf_patients_patho_idf.crs)

##### Repartition par departement

In [1428]:
patients_counts_exp1_5km_by_dept = gdf_1_5km.groupby('CODE_DEPT').size().reset_index(name='exposes')
patients_counts_exp1_5km_by_dept

,CODE_DEPT,exposes
0,75,5833
1,77,880
2,78,3381
3,91,1084
4,92,6853
5,93,2108
6,94,1996
7,95,1498


In [1429]:
patients_counts_exp1_5km_by_dept['CODE_DEPT'] = patients_counts_exp1_5km_by_dept['CODE_DEPT'].astype(str)

In [1430]:
stats_routes_1_5km = patients_counts_by_dept.merge(patients_counts_exp1_5km_by_dept, on='CODE_DEPT')
stats_routes_1_5km['pourcentage_1_5km'] = (((stats_routes_1_5km['exposes'] / stats_routes_1_5km['patient_co'])*100).round(1))
stats_routes_1_5km = stats_routes_1_5km.sort_values(by ='CODE_DEPT')
stats_routes_1_5km

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_1_5km
0,75,PARIS,10892,5833,53.6
1,77,SEINE-ET-MARNE,3018,880,29.2
2,78,YVELINES,8628,3381,39.2
3,91,ESSONNE,2992,1084,36.2
4,92,HAUTS-DE-SEINE,10174,6853,67.4
5,93,SEINE-SAINT-DENIS,3204,2108,65.8
6,94,VAL-DE-MARNE,3038,1996,65.7
7,95,VAL-D'OISE,2824,1498,53.0


In [1431]:
fig = px.bar(stats_routes_1_5km, x='NOM_DEPT', y='pourcentage_1_5km', title='Patientèle située a une distance maximale d\'1.5 km des voies routières principales par departement',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_1_5km': 'patients (%)'})
fig.show()

#### 2- Repartition par pathologie dans cette zone

In [1432]:
gdf_idf_hemato_1500m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_routes_1_5km, how='inner', predicate='intersects')
gdf_idf_hemato_1500m_dept = gdf_idf_hemato_1500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1500m')
gdf_idf_hemato_1500m = gdf_idf_hemato_1500m_dept .merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_1500m['pourcentage']= (gdf_idf_hemato_1500m['nbre_patients_<=1500m']/gdf_idf_hemato_1500m['nb_patients_hemato']*100).round(1)



gdf_idf_sein_1500m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_routes_1_5km, how='inner', predicate='intersects')
gdf_idf_sein_1500m_dept = gdf_idf_sein_1500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1500m')
gdf_idf_sein_1500m = gdf_idf_sein_1500m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_1500m['pourcentage']= (gdf_idf_sein_1500m['nbre_patients_<=1500m']/gdf_idf_sein_1500m['nb_patients_sein']*100).round(1)


gdf_idf_uro_1500m = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_routes_1_5km, how='inner', predicate='intersects')
gdf_idf_uro_1500m_dept = gdf_idf_uro_1500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1500m')
gdf_idf_uro_1500m = gdf_idf_uro_1500m_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_1500m['pourcentage']= (gdf_idf_uro_1500m['nbre_patients_<=1500m']/gdf_idf_uro_1500m['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_1500m = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_routes_1_5km, how='inner', predicate='intersects')
gdf_idf_ophtalmo_1500m_dept = gdf_idf_ophtalmo_1500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=1500m')
gdf_idf_ophtalmo_1500m = gdf_idf_ophtalmo_1500m_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_1500m['pourcentage'] = (gdf_idf_ophtalmo_1500m['nbre_patients_<=1500m']/gdf_idf_ophtalmo_1500m['nb_patients_ophtalmo']*100).round(1)

print(gdf_idf_hemato_1500m)
print(gdf_idf_sein_1500m)
print(gdf_idf_uro_1500m)
print(gdf_idf_ophtalmo_1500m)

  CODE_DEPT  nbre_patients_<=1500m  nb_patients_hemato           NOM_DEPT  \
0        75                    196                 357              PARIS   
1        77                     21                  90     SEINE-ET-MARNE   
2        78                    122                 307           YVELINES   
3        91                     41                 101            ESSONNE   
4        92                    491                 717     HAUTS-DE-SEINE   
5        93                     56                  91  SEINE-SAINT-DENIS   
6        94                     69                  93       VAL-DE-MARNE   
7        95                     49                  95         VAL-D'OISE   

   pourcentage  
0         54.9  
1         23.3  
2         39.7  
3         40.6  
4         68.5  
5         61.5  
6         74.2  
7         51.6  
  CODE_DEPT  nbre_patients_<=1500m  nb_patients_sein           NOM_DEPT  \
0        75                   3641              6754              PARIS   
1  

In [1433]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_1500m['NOM_DEPT'], y=gdf_idf_hemato_1500m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_1500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_1500m['NOM_DEPT'], y=gdf_idf_sein_1500m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_1500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_1500m['NOM_DEPT'], y=gdf_idf_uro_1500m['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_1500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_1500m['NOM_DEPT'], y=gdf_idf_ophtalmo_1500m['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_1500m['pourcentage'],
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située a une distance maximale d\' 1.5 km des routes principales par departement en fonction de leur groupe pathologique')

fig.show()

### PERIMETRE DE 2KM

In [1434]:
gdf_buffer_routes_2km = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_2km.shp')
gdf_buffer_routes_2km = gdf_buffer_routes_2km.to_crs(gdf_patients_patho_idf.crs)

In [1435]:
gdf_idf_hemato_2km = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_routes_2km, how='inner', predicate='intersects')
gdf_idf_hemato_2km_dept = gdf_idf_hemato_2km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2km')
gdf_idf_hemato_2km = gdf_idf_hemato_2km_dept .merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_2km['pourcentage']= (gdf_idf_hemato_2km['nbre_patients_<=2km']/gdf_idf_hemato_2km['nb_patients_hemato']*100).round(1)



gdf_idf_sein_2km = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_routes_2km, how='inner', predicate='intersects')
gdf_idf_sein_2km_dept = gdf_idf_sein_2km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2km')
gdf_idf_sein_2km = gdf_idf_sein_2km_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_2km['pourcentage']= (gdf_idf_sein_2km['nbre_patients_<=2km']/gdf_idf_sein_2km['nb_patients_sein']*100).round(1)


gdf_idf_uro_2km = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_routes_2km, how='inner', predicate='intersects')
gdf_idf_uro_2km_dept = gdf_idf_uro_2km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2km')
gdf_idf_uro_2km = gdf_idf_uro_2km_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_2km['pourcentage']= (gdf_idf_uro_2km['nbre_patients_<=2km']/gdf_idf_uro_2km['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_2km = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_routes_2km, how='inner', predicate='intersects')
gdf_idf_ophtalmo_2km_dept = gdf_idf_ophtalmo_2km.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2km')
gdf_idf_ophtalmo_2km = gdf_idf_ophtalmo_2km_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_2km['pourcentage'] = (gdf_idf_ophtalmo_2km['nbre_patients_<=2km']/gdf_idf_ophtalmo_2km['nb_patients_ophtalmo']*100).round(1)

print(gdf_idf_hemato_2km)
print(gdf_idf_sein_2km)
print(gdf_idf_uro_2km)
print(gdf_idf_ophtalmo_2km)

  CODE_DEPT  nbre_patients_<=2km  nb_patients_hemato           NOM_DEPT  \
0        75                  242                 357              PARIS   
1        77                   28                  90     SEINE-ET-MARNE   
2        78                  152                 307           YVELINES   
3        91                   50                 101            ESSONNE   
4        92                  602                 717     HAUTS-DE-SEINE   
5        93                   68                  91  SEINE-SAINT-DENIS   
6        94                   77                  93       VAL-DE-MARNE   
7        95                   63                  95         VAL-D'OISE   

   pourcentage  
0         67.8  
1         31.1  
2         49.5  
3         49.5  
4         84.0  
5         74.7  
6         82.8  
7         66.3  
  CODE_DEPT  nbre_patients_<=2km  nb_patients_sein           NOM_DEPT  \
0        75                 4636              6754              PARIS   
1        77              

In [1436]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_2km['NOM_DEPT'], y=gdf_idf_hemato_2km['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_2km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_2km['NOM_DEPT'], y=gdf_idf_sein_2km['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_2km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_2km['NOM_DEPT'], y=gdf_idf_uro_2km['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_2km['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_2km['NOM_DEPT'], y=gdf_idf_ophtalmo_2km['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_2km['pourcentage'],
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située a une distance maximale de 2 km des routes principales par departement en fonction de leur groupe pathologique')

fig.show()

### PERIMETRE DE 2.5km

#### 1- Repartition de l'ensemble des patients dans cette zone

##### Pourcentage dans l'ensemble de la région d'IDF

In [1437]:
len(gdf_2_5km.drop_duplicates(subset='pseudo_provisoire'))

34076

In [1438]:
pourcentage = len(gdf_2_5km.drop_duplicates(subset='pseudo_provisoire'))/len(df_patients.drop_duplicates(subset='pseudo_provisoire'))*100
pourcentage.__round__(2)
print(f"{pourcentage:.2f}% de patients sont situés à moins de 2.5km des voies routières en IDF")

76.11% de patients sont situés à moins de 2.5km des voies routières en IDF


##### Repartition par département

In [1439]:
patients_counts_exp2_5_by_dept = gdf_2_5km.groupby('CODE_DEPT').size().reset_index(name='exposes')
patients_counts_exp2_5_by_dept

,CODE_DEPT,exposes
0,75,8547
1,77,1388
2,78,5344
3,91,1569
4,92,9580
5,93,2799
6,94,2626
7,95,2223


In [1440]:
patients_counts_exp2_5_by_dept['CODE_DEPT'] = patients_counts_exp2_5_by_dept['CODE_DEPT'].astype(str)

In [1441]:
stats_routes_2_5km = patients_counts_by_dept.merge(patients_counts_exp2_5_by_dept, on='CODE_DEPT')
stats_routes_2_5km['pourcentage_2_5km'] = (((stats_routes_2_5km['exposes'] / stats_routes_2_5km['patient_co'])*100).round(1))
stats_routes_2_5km = stats_routes_2_5km.sort_values(by ='CODE_DEPT')
stats_routes_2_5km

,CODE_DEPT,NOM_DEPT,patient_co,exposes,pourcentage_2_5km
0,75,PARIS,10892,8547,78.5
1,77,SEINE-ET-MARNE,3018,1388,46.0
2,78,YVELINES,8628,5344,61.9
3,91,ESSONNE,2992,1569,52.4
4,92,HAUTS-DE-SEINE,10174,9580,94.2
5,93,SEINE-SAINT-DENIS,3204,2799,87.4
6,94,VAL-DE-MARNE,3038,2626,86.4
7,95,VAL-D'OISE,2824,2223,78.7


In [1442]:
fig = px.bar(stats_routes_2_5km, x='NOM_DEPT', y='pourcentage_2_5km', title='Patientèle située a une distance maximale de 500m des voies routières principales par departement',
             labels={'NOM_DEPT': 'Départment', 'pourcentage_2_5km': 'patients (%)'})
fig.show()

#### 2- Repartition par pathologie cette zone

In [1443]:
gdf_buffer_routes_2500m = gpd.read_file('../geocodeur/from_hegp/data/zones_geographiques/CARTO_QGIS/highways/highways_2_5km.shp')
gdf_buffer_routes_2500m = gdf_buffer_routes_2500m.to_crs(gdf_patients_patho_idf.crs)

In [1444]:
gdf_idf_hemato_2500m = gpd.sjoin(gdf_idf_hemato_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_hemato_2500m_dept = gdf_idf_hemato_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2.5km')
gdf_idf_hemato_2500m = gdf_idf_hemato_2500m_dept .merge(gdf_idf_hemato_dept, on='CODE_DEPT')
gdf_idf_hemato_2500m['pourcentage']= (gdf_idf_hemato_2500m['nbre_patients_<=2.5km']/gdf_idf_hemato_2500m['nb_patients_hemato']*100).round(1)



gdf_idf_sein_2500m = gpd.sjoin(gdf_idf_sein_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_sein_2500m_dept = gdf_idf_sein_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2.5km')
gdf_idf_sein_2500m = gdf_idf_sein_2500m_dept.merge(gdf_idf_sein_dept, on='CODE_DEPT')
gdf_idf_sein_2500m['pourcentage']= (gdf_idf_sein_2500m['nbre_patients_<=2.5km']/gdf_idf_sein_2500m['nb_patients_sein']*100).round(1)


gdf_idf_uro_2500m = gpd.sjoin(gdf_idf_uro_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_uro_2500m_dept = gdf_idf_uro_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2.5km')
gdf_idf_uro_2500m = gdf_idf_uro_2500m_dept.merge(gdf_idf_uro_dept, on='CODE_DEPT')
gdf_idf_uro_2500m['pourcentage']= (gdf_idf_uro_2500m['nbre_patients_<=2.5km']/gdf_idf_uro_2500m['nb_patients_uro']*100).round(1)


gdf_idf_ophtalmo_2500m = gpd.sjoin(gdf_idf_ophtalmo_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_ophtalmo_2500m_dept = gdf_idf_ophtalmo_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2.5km')
gdf_idf_ophtalmo_2500m = gdf_idf_ophtalmo_2500m_dept.merge(gdf_idf_ophtamo_dept, on='CODE_DEPT')
gdf_idf_ophtalmo_2500m['pourcentage'] = (gdf_idf_ophtalmo_2500m['nbre_patients_<=2.5km']/gdf_idf_ophtalmo_2500m['nb_patients_ophtalmo']*100).round(1)

gdf_idf_thorax_2500m = gpd.sjoin(gdf_idf_thorax_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_thorax_2500m_dept = gdf_idf_thorax_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_thorax_2500m = gdf_idf_thorax_2500m_dept.merge(gdf_idf_thorax_dept, on='CODE_DEPT')
gdf_idf_thorax_2500m['pourcentage'] = (gdf_idf_thorax_2500m['nbre_patients_<=2500m']/gdf_idf_thorax_2500m['nb_patients_thorax']*100).round(1)


gdf_idf_gastro_2500m = gpd.sjoin(gdf_idf_gastro_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_gastro_2500m_dept = gdf_idf_gastro_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_gastro_2500m  = gdf_idf_gastro_2500m_dept.merge(gdf_idf_gastro_dept, on='CODE_DEPT')
gdf_idf_gastro_2500m ['pourcentage'] = (gdf_idf_gastro_2500m['nbre_patients_<=2500m']/gdf_idf_gastro_2500m['nb_patients_gastro']*100).round(1)

gdf_idf_gyneco_2500m  = gpd.sjoin(gdf_idf_gyneco_points, gdf_buffer_routes_2500m , how='inner', predicate='intersects')
gdf_idf_gyneco_2500m_dept = gdf_idf_gyneco_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_gyneco_2500m = gdf_idf_gyneco_2500m_dept.merge(gdf_idf_gyneco_dept, on='CODE_DEPT')
gdf_idf_gyneco_2500m['pourcentage'] = (gdf_idf_gyneco_2500m['nbre_patients_<=2500m']/gdf_idf_gyneco_2500m['nb_patients_gyneco']*100).round(1)

gdf_idf_sarcome_2500m = gpd.sjoin(gdf_idf_sarcome_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_sarcome_2500m_dept = gdf_idf_sarcome_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_sarcome_2500m = gdf_idf_sarcome_2500m_dept.merge(gdf_idf_sarcome_dept, on='CODE_DEPT')
gdf_idf_sarcome_2500m['pourcentage'] = (gdf_idf_sarcome_2500m['nbre_patients_<=2500m']/gdf_idf_sarcome_2500m['nb_patients_sarcome']*100).round(1)

gdf_idf_orl_2500m = gpd.sjoin(gdf_idf_orl_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_orl_2500m_dept = gdf_idf_orl_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_orl_2500m = gdf_idf_orl_2500m_dept.merge(gdf_idf_orl_dept, on='CODE_DEPT')
gdf_idf_orl_2500m['pourcentage'] = (gdf_idf_orl_2500m['nbre_patients_<=2500m']/gdf_idf_orl_2500m['nb_patients_orl']*100).round(1)

gdf_idf_dermato_2500m = gpd.sjoin(gdf_idf_dermato_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_dermato_2500m_dept = gdf_idf_dermato_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_dermato_2500m= gdf_idf_dermato_2500m_dept.merge(gdf_idf_dermato_dept, on='CODE_DEPT')
gdf_idf_dermato_2500m['pourcentage'] = (gdf_idf_dermato_2500m['nbre_patients_<=2500m']/gdf_idf_dermato_2500m['nb_patients_dermato']*100).round(1)

gdf_idf_endocrino_2500m = gpd.sjoin(gdf_idf_endocrino_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_endocrino_2500m_dept = gdf_idf_endocrino_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_endocrino_2500m = gdf_idf_endocrino_2500m_dept.merge(gdf_idf_endocrino_dept, on='CODE_DEPT')
gdf_idf_endocrino_2500m['pourcentage'] = (gdf_idf_endocrino_2500m['nbre_patients_<=2500m']/gdf_idf_endocrino_2500m['nb_patients_endocrino']*100).round(1)

gdf_idf_neuro_2500m = gpd.sjoin(gdf_idf_neuro_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_neuro_2500m_dept = gdf_idf_neuro_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_neuro_2500m  = gdf_idf_neuro_2500m_dept.merge(gdf_idf_neuro_dept, on='CODE_DEPT')
gdf_idf_neuro_2500m['pourcentage'] = (gdf_idf_neuro_2500m['nbre_patients_<=2500m']/gdf_idf_neuro_2500m['nb_patients_neuro']*100).round(1)

gdf_idf_autre_2500m = gpd.sjoin(gdf_idf_autre_points, gdf_buffer_routes_2500m, how='inner', predicate='intersects')
gdf_idf_autre_2500m_dept = gdf_idf_autre_2500m.groupby('CODE_DEPT').size().reset_index(name='nbre_patients_<=2500m')
gdf_idf_autre_2500m = gdf_idf_autre_2500m_dept.merge(gdf_idf_autre_dept, on='CODE_DEPT')
gdf_idf_autre_2500m['pourcentage'] = (gdf_idf_autre_2500m['nbre_patients_<=2500m']/gdf_idf_autre_2500m['nb_patients_autre']*100).round(1)

print(gdf_idf_hemato_2500m)
print(gdf_idf_sein_2500m)
print(gdf_idf_uro_2500m)
print(gdf_idf_ophtalmo_2500m)

  CODE_DEPT  nbre_patients_<=2.5km  nb_patients_hemato           NOM_DEPT  \
0        75                    279                 357              PARIS   
1        77                     33                  90     SEINE-ET-MARNE   
2        78                    189                 307           YVELINES   
3        91                     56                 101            ESSONNE   
4        92                    689                 717     HAUTS-DE-SEINE   
5        93                     78                  91  SEINE-SAINT-DENIS   
6        94                     83                  93       VAL-DE-MARNE   
7        95                     76                  95         VAL-D'OISE   

   pourcentage  
0         78.2  
1         36.7  
2         61.6  
3         55.4  
4         96.1  
5         85.7  
6         89.2  
7         80.0  
  CODE_DEPT  nbre_patients_<=2.5km  nb_patients_sein           NOM_DEPT  \
0        75                   5367              6754              PARIS   
1  

In [1445]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_hemato_2500m['NOM_DEPT'], y=gdf_idf_hemato_2500m['pourcentage'], name='cancer hemato',
                     text=gdf_idf_hemato_2500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sein_2500m['NOM_DEPT'], y=gdf_idf_sein_2500m['pourcentage'], name='cancer sein',
                     text=gdf_idf_sein_2500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_uro_2500m['NOM_DEPT'], y=gdf_idf_uro_2500m['pourcentage'], name='cancer uro',
                     text=gdf_idf_uro_2500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_ophtalmo_2500m['NOM_DEPT'], y=gdf_idf_ophtalmo_2500m['pourcentage'], name='cancer ophtalmo',
                     text=gdf_idf_ophtalmo_2500m['pourcentage'],
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_thorax_2500m['NOM_DEPT'], y=gdf_idf_thorax_2500m['pourcentage'], name='cancer thorax',
                     text=gdf_idf_thorax_2500m['pourcentage'].round(1),
                     textposition='outside'))

fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle située a une distance maximale de 2.5 km des routes principales par departement en fonction de leur groupe pathologique')

fig.show()

In [1451]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_gastro_2500m['NOM_DEPT'], y=gdf_idf_gastro_2500m['pourcentage'], name='cancer gastro',
                     text=gdf_idf_gastro_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_gyneco_2500m['NOM_DEPT'], y=gdf_idf_gyneco_2500m['pourcentage'], name='cancer gyneco',
                     text=gdf_idf_gyneco_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_sarcome_2500m['NOM_DEPT'], y=gdf_idf_sarcome_2500m['pourcentage'], name='cancer sarcome',
                     text=gdf_idf_sarcome_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_orl_2500m['NOM_DEPT'], y=gdf_idf_orl_2500m['pourcentage'], name='cancer orl',
                     text=gdf_idf_orl_2500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 2500m des autoroutes en fonction de leur groupe pathologique')

fig.show()

In [1450]:
# Création d'une figure avec des sous-trames
fig =  go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=gdf_idf_dermato_2500m['NOM_DEPT'], y=gdf_idf_dermato_2500m['pourcentage'], name='cancer dermato',
                     text=gdf_idf_dermato_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_endocrino_2500m['NOM_DEPT'], y=gdf_idf_endocrino_2500m['pourcentage'], name='cancer endocrino',
                     text=gdf_idf_endocrino_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_neuro_2500m['NOM_DEPT'], y=gdf_idf_neuro_2500m['pourcentage'], name='cancer neuro',
                     text=gdf_idf_neuro_2500m['pourcentage'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=gdf_idf_autre_2500m['NOM_DEPT'], y=gdf_idf_autre_2500m['pourcentage'], name='cancer autre',
                     text=gdf_idf_autre_2500m['pourcentage'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle repartie par département et située à une distance maximale de 2500m des autoroutes en fonction de leur groupe pathologique')

fig.show()

## TABLEAU RECAPITULATIF DE L'ENSEMBLE DE LA PATIENTELE EN FONCTION DE LEUR PROXIMITE AUX VOIES ROUTIERES PRINCIPALES

In [1448]:
# Création d'une figure avec des sous-trames
fig = fig = go.Figure()

# Ajout des bar plots pour chaque périmètre
fig.add_trace(go.Bar(x=stats_routes_150m['NOM_DEPT'], y=stats_routes_150m['pourcentage_150'], name='périmètre 150m',
                     text=stats_routes_150m['pourcentage_150'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=stats_routes_500m['NOM_DEPT'], y=stats_routes_500m['pourcentage_500'], name='périmètre 500m',
                     text=stats_routes_500m['pourcentage_500'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=stats_routes_1km['NOM_DEPT'], y=stats_routes_1km['pourcentage_1km'], name='périmètre 1000m',
                     text=stats_routes_1km['pourcentage_1km'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=stats_routes_1_5km['NOM_DEPT'], y=stats_routes_1_5km['pourcentage_1_5km'], name='périmètre 1500m',
                     text=stats_routes_1_5km['pourcentage_1_5km'].round(1),
                     textposition='outside'))
fig.add_trace(go.Bar(x=stats_routes_2_5km['NOM_DEPT'], y=stats_routes_2_5km['pourcentage_2_5km'], name='périmètre 2500m',
                     text=stats_routes_2_5km['pourcentage_2_5km'].round(1),
                     textposition='outside'))


fig.update_layout(barmode='group', 
                  xaxis=dict(title='Départment'),
                  yaxis=dict(title='% Patients'),
                  title='Patientèle par département en fonction de leur proximité aux voies routières principales')

fig.show()